In [ ]:
from dj_notebook import activate

plus = activate()


Output()

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from io import BytesIO
from IPython.display import display, IFrame
from cotizaciones.functions import generar_pdf_cotizacion 


In [ ]:

# 2) Datos de prueba
ejemplo = {
    'numero_cotizacion': '0001-2025',
    'fecha_vencimiento': '2025-05-15',
    'estado': 'PENDIENTE',
    'total_estimado': '$1.234.567',
    'items': [
        {'descripcion': 'Sensor X', 'cantidad': 2, 'precio_unitario': 123.45, 'costo_total': 246.90},
        {'descripcion': 'Actuador Y', 'cantidad': 1, 'precio_unitario': 987.65, 'costo_total': 987.65},
    ],
    'observaciones': 'Este es un texto de prueba para observar el wrapping en el área de comentarios.'
}

# 3) Generar bytes del PDF
pdf_bytes = generar_pdf_cotizacion(
    datos_cotizacion=ejemplo,
    nombre_empresa='Mi Empresa S.A.',
    rut_empresa='12.345.678-9',
    direccion_empresa='Av. Siempre Viva 123',
    telefono_empresa='+56 9 8765 4321',
    email_empresa='contacto@miempresa.cl',
    sitio_web_empresa='www.miempresa.cl',
    nombre_cliente='Cliente de Prueba',
    telefono_cliente='+56 2 2345 6789',
    direccion_cliente='Calle Falsa 456',
    rut_cliente='98.765.432-1',
    email_cliente='cliente@prueba.com'
)

# 4) Guardar en un archivo temporal y mostrar inline
path_temporal = 'temp_cotizacion.pdf'
with open(path_temporal, 'wb') as f:
    f.write(pdf_bytes)

display(IFrame(src=path_temporal, width=1024, height=768))

In [ ]:
from io import BytesIO
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle
from reportlab.lib.utils import ImageReader
from textwrap import wrap
import base64


def generar_pdf_cotizacion(
    datos_cotizacion,
    logo_base64,
    nombre_empresa, rut_empresa, direccion_empresa,
    telefono_empresa, email_empresa, sitio_web_empresa,
    nombre_cliente, rut_cliente, direccion_cliente,
    destinatarios,
    texto_introduccion="Ud. ha solicitado información sobre los precios de",
    texto_introduccion2=", a continuación, aparece nuestra cotización:\r\n",
    items=None,
    observaciones=None,
    cierre=None,
    firmante=None,
    cargo=None,
    contactos=None,
    ubicacion="Santiago"
):
    """
    Genera una cotización en PDF con formato A4, encabezado con logo a la derecha,
    fecha a la izquierda, pie de página centrado y firma centrada.
    Se muestran dos textos de introducción entre los cuales aparece la descripción
    de la cotización (campo 'descripcion_cotizacion' en datos_cotizacion).
    """
    if items is None:
        items = []
    if contactos is None:
        contactos = []

    buffer = BytesIO()
    pdf = canvas.Canvas(buffer, pagesize=A4)
    ancho, alto = A4
    margen_x, margen_y = 40, 40

    # Fecha y lugar
    fecha_str = datos_cotizacion.get('fecha_cotizacion', datetime.now().strftime("%d de %B de %Y"))
    pdf.setFont("Helvetica", 9)
    pdf.drawString(margen_x, alto - margen_y + 10, f"{ubicacion}, {fecha_str}")

    # Logo a la derecha
    if logo_base64:
        try:
            b64 = logo_base64.split(',', 1)[1] if ',' in logo_base64 else logo_base64
            logo_data = base64.b64decode(b64)
            img = ImageReader(BytesIO(logo_data))
            iw, ih = img.getSize()
            aspect = ih / float(iw)
            w_logo = 120
            h_logo = w_logo * aspect
            x_logo = ancho - margen_x - w_logo
            pdf.drawImage(img, x_logo, alto - margen_y - h_logo,
                          width=w_logo, height=h_logo, mask='auto')
        except:
            pass

    # Título
    y = alto - margen_y - 20
    pdf.setFont("Helvetica-Bold", 14)
    pdf.drawCentredString(ancho/2, y, f"Cotización Productos N° {datos_cotizacion['numero_cotizacion']}")
    y -= 30

    # Datos del cliente
    pdf.setFont("Helvetica-Bold", 10)
    pdf.drawString(margen_x, y, f"Cliente: {nombre_cliente}")
    y -= 14
    pdf.setFont("Helvetica", 10)
    pdf.drawString(margen_x, y, f"Rut: {rut_cliente}")
    y -= 14
    pdf.drawString(margen_x, y, f"Dirección: {direccion_cliente}")
    y -= 20

    # Saludo
    pdf.setFont("Helvetica-Bold", 10)
    pdf.drawString(margen_x, y, f"Estimado/a: {destinatarios}")
    y -= 20

    # Texto de introducción parte 1
    text = pdf.beginText(margen_x, y)
    text.setFont("Helvetica", 10)
    for line in wrap(texto_introduccion, width=90):
        text.textLine(line)
        y -= 12
    pdf.drawText(text)
    y -= 10

    # Descripción de la cotización
    descripcion = datos_cotizacion.get('descripcion_cotizacion', '')
    if descripcion:
        text = pdf.beginText(margen_x, y)
        pdf.setFont("Helvetica-Bold", 10)
        for line in wrap(descripcion, width=90):
            text.textLine(line)
            y -= 12
        pdf.drawText(text)
        y -= 10

    # Texto de introducción parte 2
    text = pdf.beginText(margen_x, y)
    text.setFont("Helvetica", 10)
    for line in wrap(texto_introduccion2, width=90):
        text.textLine(line)
        y -= 12
    pdf.drawText(text)
    y -= 20

    # Tabla de ítems
    data = [["Descripción", "Cantidad", "Valor Unit", "Valor Total Neto"]]
    for it in items:
        data.append([it['descripcion'], str(it['cantidad']), it['valor_unitario'], it['valor_total_neto']])
    table_width = ancho - 2*margen_x
    colWidths = [0.4*table_width, 0.15*table_width, 0.2*table_width, 0.25*table_width]
    tabla = Table(data, colWidths=colWidths)
    tabla.setStyle(TableStyle([
        ('BACKGROUND',(0,0),(-1,0),colors.lightgrey),
        ('FONTNAME',(0,0),(-1,0),'Helvetica-Bold'),
        ('ALIGN',(1,1),(-1,-1),'RIGHT'),
        ('GRID',(0,0),(-1,-1),0.5,colors.black),
        ('FONTSIZE',(0,0),(-1,-1),10)
    ]))
    w_t, h_t = tabla.wrapOn(pdf,0,0)
    tabla.drawOn(pdf, margen_x, y - h_t)
    y -= h_t + 20

    # Texto fijo antes de observaciones
    fijo1 = "Valores netos expresados en USD, conversión del dólar, observado del día de la compra +$5"
    pdf.setFont("Helvetica-Oblique", 9)
    for line in wrap(fijo1, width=90):
        pdf.drawString(margen_x, y, line)
        y -= 12
    fijo2 = "Cotización válida por 1 semana"
    pdf.setFont("Helvetica", 10)
    for line in wrap(fijo2, width=90):
        pdf.drawString(margen_x, y, line)
        y -= 14
    y -= 10

    # Observaciones
    if observaciones:
        pdf.setFont("Helvetica-Oblique", 9)
        text = pdf.beginText(margen_x, y)
        for line in wrap(observaciones, width=90):
            text.textLine(line)
            y -= 12
        pdf.drawText(text)
        y -= 20

    # Cierre
    cierre_base = (
        "Gracias por darnos la oportunidad de ofrecerle este presupuesto. "
        "Como siempre, es para nosotros un placer hacer negocios con ustedes. "
        "Esperamos hacer realidad este pedido para su completa satisfacción."
    )
    pdf.setFont("Helvetica", 10)
    text = pdf.beginText(margen_x, y)
    for line in wrap(cierre_base, width=90):
        text.textLine(line)
        y -= 12
    pdf.drawText(text)
    y -= 30

    # Firma
    sig = ['Atentamente,', firmante]
    if cargo:
        sig.append(cargo)
    for i, line in enumerate(sig):
        font = 'Helvetica-Bold' if i == 0 else 'Helvetica'
        size = 10 if i <= 1 else 9
        pdf.setFont(font, size)
        pdf.drawCentredString(ancho/2, y - i*14, line)
    y -= len(sig)*14 + 10

    # Pie de página
    footer_y = margen_y - 10
    pdf.setFont("Helvetica", 8)
    for idx, link in enumerate(contactos):
        yy = footer_y + (len(contactos)-idx)*10
        pdf.drawCentredString(ancho/2, yy, link)
    page = pdf.getPageNumber()
    pdf.drawCentredString(ancho/2, footer_y-10, f"{page}/{page}")

    pdf.showPage()
    pdf.save()
    buffer.seek(0)
    return buffer.getvalue()


In [12]:
# 1) Auto-reload para ver cambios al vuelo
%load_ext autoreload
%autoreload 2

# 2) Imports
from io import BytesIO
from IPython.display import display, IFrame


# 3) Datos de prueba
ejemplo_datos = {
    'numero_cotizacion': '789',
    'fecha_cotizacion': '25 de Abril de 2025'
}

ejemplo_items = [
    {
        'descripcion': 'Asus P1403CVA-S60607X\n'
                       'Intel core i5-13420H\n'
                       '512 GB SSD\n'
                       '16 GB RAM\n'
                       'Monitor 14\n'
                       'Windows 11 Pro',
        'cantidad': 1,
        'valor_unitario': '791 USD',
        'valor_total_neto': '791 USD'
    },
    {
        'descripcion': 'Lenovo ThinkBook 14\n'
                       'Intel core i5-13420H\n'
                       '8GB RAM\n'
                       '512GB SSD\n'
                       '14 inch\n'
                       'Windows 11 Pro\n'
                       '1920 x 1200 LCD',
        'cantidad': 1,
        'valor_unitario': '970 USD',
        'valor_total_neto': '970 USD'
    }
]

ejemplo_contactos = [
    'Snabb IT | Una empresa del Grupo A&G',
    'Av. José Miguel Carrera N° 3840 Of- 808, San Miguel – Fono: (56-2) 27596140',
    'email: lrojas@snabb-it.cl administracion@aygasociados.cl',
    'Visítenos en https://snabbit.cl'
]

# 4) Generar y mostrar el PDF
pdf_bytes = generar_pdf_cotizacion(
    datos_cotizacion=ejemplo_datos,
    nombre_empresa='Snabb IT | Una empresa del Grupo A&G',
    rut_empresa='',
    direccion_empresa='Av. José Miguel Carrera N° 3840 Of- 808, San Miguel',
    telefono_empresa='(56-2) 27596140',
    email_empresa='lrojas@snabb-it.cl administracion@aygasociados.cl',
    sitio_web_empresa='https://snabbit.cl',
    nombre_cliente='Molina Rios Abogados',
    rut_cliente='',
    direccion_cliente='',
    destinatario='Felipe Correa / Carlos Molina',
    texto_introduccion=(
        'Ud. ha solicitado información sobre los precios Notebooks, '
        'a continuación, aparece nuestra cotización:'
    ),
    items=ejemplo_items,
    observaciones=(
        'Valores netos expresados en USD, conversión del dólar, '
        'observado del día de la compra +$5\n'
        'Cotización válida por 1 semana'
    ),
    cierre=(
        'Gracias por darnos la oportunidad de ofrecerle este presupuesto. '
        'Como siempre, es para nosotros un placer hacer negocios con ustedes. '
        'Esperamos hacer realidad este pedido para su completa satisfacción.'
    ),
    firmante='Luis Rojas Molina',
    cargo='Gerente de Tecnologías',
    contactos=ejemplo_contactos
)

# Guardar en archivo temporal y mostrar inline
with open('temp_cotizacion.pdf', 'wb') as f:
    f.write(pdf_bytes)

display(IFrame('temp_cotizacion.pdf', width=1024, height=768))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


TypeError: generar_pdf_cotizacion() got an unexpected keyword argument 'destinatario'

In [ ]:
from io import BytesIO
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle
from reportlab.lib.utils import ImageReader
from textwrap import wrap
import base64

# ——— Constantes de estilo ———
ESTILOS = {
    'titulo':       ('Helvetica-Bold', 16, colors.HexColor('#2E86C1')),
    'subtitulo':    ('Helvetica-Bold', 12, colors.black),
    'normal':       ('Helvetica',       10, colors.black),
    'pequeno':      ('Helvetica',        9, colors.black),
    'fondo_enc':    colors.lightgrey,
    'zebra1':       colors.whitesmoke,
    'zebra2':       colors.white,
}

# ——— Funciones de dibujo ———
def dibujar_encabezado(pdf, ubicacion, fecha_str, logo_base64, ancho, alto, mx, my):
    # Fecha y lugar
    pdf.setFont(*ESTILOS['pequeno'][:2])
    pdf.setFillColor(ESTILOS['pequeno'][2])
    pdf.drawString(mx, alto - my + 10, f"{ubicacion}, {fecha_str}")
    # Logo
    if logo_base64:
        try:
            b64 = logo_base64.split(',',1)[1] if ',' in logo_base64 else logo_base64
            img = ImageReader(BytesIO(base64.b64decode(b64)))
            iw, ih = img.getSize()
            w_logo = 120
            h_logo = w_logo * (ih/iw)
            pdf.drawImage(img, ancho-mx-w_logo, alto-my-h_logo,
                          width=w_logo, height=h_logo, mask='auto')
        except:
            pass
    # Línea separadora
    pdf.setStrokeColor(colors.grey)
    pdf.setLineWidth(0.5)
    pdf.line(mx, alto-my-60, ancho-mx, alto-my-60)

def dibujar_datos_cliente(pdf, nombre, rut, direccion, destinatarios, mx, y):
    pdf.setFont(*ESTILOS['subtitulo'][:2])
    pdf.drawString(mx, y, f"Cliente: {nombre}")
    y -= 14
    pdf.setFont(*ESTILOS['normal'][:2])
    pdf.drawString(mx, y, f"RUT: {rut}")
    y -= 14
    pdf.drawString(mx, y, f"Dirección: {direccion}")
    y -= 20
    pdf.setFont(*ESTILOS['subtitulo'][:2])
    pdf.drawString(mx, y, f"Estimado/a: {destinatarios}")
    return y - 20

def dibujar_introduccion(pdf, intro1, descripcion, intro2, mx, y):
    texto = f"{intro1} {descripcion}{intro2}"
    text = pdf.beginText(mx, y)
    pdf.setFont(*ESTILOS['normal'][:2])
    for line in wrap(texto, width=90):
        text.textLine(line)
        y -= 12
    pdf.drawText(text)
    return y - 20

def dibujar_tabla_items(pdf, items, ancho, mx, y):
    # Construir data
    data = [["Descripción", "Cantidad", "Valor Unit", "Total Neto"]]
    for it in items:
        data.append([it['descripcion'], str(it['cantidad']),
                     it['valor_unitario'], it['valor_total_neto']])
    # TableStyle con zebra stripes
    style = TableStyle([
        ('BACKGROUND',(0,0),(-1,0), ESTILOS['fondo_enc']),
        ('FONTNAME',(0,0),(-1,0),'Helvetica-Bold'),
        ('ALIGN',(1,1),(-1,-1),'RIGHT'),
        ('GRID',(0,0),(-1,-1),0.5,colors.black),
        ('FONTSIZE',(0,0),(-1,-1),10),
        ('BOTTOMPADDING',(0,0),(-1,-1),6),
        ('TOPPADDING',(0,0),(-1,-1),6),
    ])
    for row in range(1, len(data)):
        bg = ESTILOS['zebra1'] if row % 2 == 0 else ESTILOS['zebra2']
        style.add('BACKGROUND', (0,row), (-1,row), bg)
    table_width = ancho - 2*mx
    colWidths = [0.4*table_width, 0.15*table_width,
                 0.2*table_width, 0.25*table_width]
    tabla = Table(data, colWidths=colWidths, style=style)
    w, h = tabla.wrapOn(pdf,0,0)
    tabla.drawOn(pdf, mx, y - h)
    return y - h - 20

def dibujar_observaciones(pdf, observaciones, mx, y):
    if not observaciones:
        return y
    pdf.setFont(*ESTILOS['pequeno'][:2])
    text = pdf.beginText(mx, y)
    for line in wrap(observaciones, width=90):
        text.textLine(line)
        y -= 12
    pdf.drawText(text)
    return y - 20

def dibujar_cierre(pdf, mx, y):
    cierre = (
        "Gracias por darnos la oportunidad de ofrecerle este presupuesto. "
        "Es un placer hacer negocios con usted. Esperamos su confirmación."
    )
    pdf.setFont(*ESTILOS['normal'][:2])
    text = pdf.beginText(mx, y)
    for line in wrap(cierre, width=90):
        text.textLine(line)
        y -= 12
    pdf.drawText(text)
    return y - 30

def dibujar_firma(pdf, firmante, cargo, ancho, y):
    sig = ['Atentamente,', firmante]
    if cargo:
        sig.append(cargo)
    for i, linea in enumerate(sig):
        font = 'Helvetica-Bold' if i==0 else 'Helvetica'
        size = 10 if i<2 else 9
        pdf.setFont(font, size)
        pdf.drawCentredString(ancho/2, y - i*14, linea)
    return y - len(sig)*14 - 10

def dibujar_pie(pdf, contactos, ancho, my):
    footer_y = my - 10
    pdf.setFont(*ESTILOS['pequeno'][:2])
    for idx, link in enumerate(contactos):
        yy = footer_y + (len(contactos)-idx)*10
        pdf.drawCentredString(ancho/2, yy, link)
    page = pdf.getPageNumber()
    pdf.drawCentredString(ancho/2, footer_y-20, f"{page}/{page}")

# ——— Función principal refactorizada ———
def generar_pdf_cotizacion(
    datos, logo_b64,
    nombre_empresa, rut_empresa, direccion_empresa,
    telefono_empresa, email_empresa, sitio_web_empresa,
    nombre_cliente, rut_cliente, direccion_cliente,
    destinatarios,
    items, observaciones, firmante, cargo, contactos,
    ubicacion="Santiago"
):
    buffer = BytesIO()
    pdf = canvas.Canvas(buffer, pagesize=A4)
    ancho, alto = A4
    mx, my = 40, 40

    fecha = datos.get('fecha_cotizacion',
                      datetime.now().strftime("%d de %B de %Y"))
    # Encabezado
    dibujar_encabezado(pdf, ubicacion, fecha, logo_b64, ancho, alto, mx, my)

    # Título
    pdf.setFont(*ESTILOS['titulo'][:2])
    pdf.setFillColor(ESTILOS['titulo'][2])
    pdf.drawCentredString(ancho/2, alto-my-80,
                          f"Cotización N° {datos['numero_cotizacion']}")

    # Datos del cliente
    y = alto - my - 110
    y = dibujar_datos_cliente(pdf, nombre_cliente, rut_cliente,
                              direccion_cliente, destinatarios, mx, y)

    # Introducción
    y = dibujar_introduccion(pdf,datos.get('descripcion_cotizacion',''), mx, y)

    # Tabla de ítems
    y = dibujar_tabla_items(pdf, items, ancho, mx, y)

    # Observaciones
    y = dibujar_observaciones(pdf, observaciones, mx, y)

    # Cierre y firma
    y = dibujar_cierre(pdf, mx, y)
    y = dibujar_firma(pdf, firmante, cargo, ancho, y)

    # Pie de página
    dibujar_pie(pdf, contactos, ancho, my)

    pdf.showPage()
    pdf.save()
    buffer.seek(0)
    return buffer.getvalue()


In [ ]:
from io import BytesIO
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle
from reportlab.lib.utils import ImageReader
from textwrap import wrap
import base64

# ——— Constantes de fuentes ———
FONTS = {
    'fecha':        ("Helvetica",        9),
    'titulo':       ("Helvetica-Bold",  14),
    'datos_label':  ("Helvetica-Bold",  10),
    'datos':        ("Helvetica",       10),
    'introduccion': ("Helvetica",       10),
    'tabla_head':   ("Helvetica-Bold",  10),
    'tabla':        ("Helvetica",       10),
    'fijo':         ("Helvetica-Oblique",9),
    'cierre':       ("Helvetica",       10),
    'firma_label':  ("Helvetica-Bold",  10),
    'firma':        ("Helvetica",       10),
    'firma_cargo':  ("Helvetica",        9),
    'footer':       ("Helvetica",        8),
}

# ——— Funciones de dibujo ———
def draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my):
    pdf.setFont(*FONTS['fecha'])
    pdf.drawString(mx, alto - my + 1, f"{ubicacion}, {fecha_str}")

def draw_logo(pdf, logo_b64, ancho, alto, mx, my):
    if not logo_b64:
        return
    try:
        b64 = logo_b64.split(',', 1)[1] if ',' in logo_b64 else logo_b64
        img = ImageReader(BytesIO(base64.b64decode(b64)))
        iw, ih = img.getSize()
        w_logo = 120
        h_logo = w_logo * (ih / iw)
        pdf.drawImage(img, ancho - mx - w_logo, alto - my - h_logo,
                      width=w_logo, height=h_logo, mask='auto')
    except Exception:
        pass

def draw_separator(pdf, mx, ancho, alto, my):
    pdf.setStrokeColor(colors.grey)
    pdf.setLineWidth(0.5)
    pdf.line(mx, alto - my - 60, ancho - mx, alto - my - 60)

def draw_encabezado(pdf, ubicacion, fecha_str, logo_b64, ancho, alto, mx, my):
    draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my)
    draw_logo(pdf, logo_b64, ancho, alto, mx, my)
    #draw_separator(pdf, mx, ancho, alto, my)

def draw_titulo(pdf, numero, ancho, alto, mx, my):
    pdf.setFont(*FONTS['titulo'])
    pdf.drawCentredString(ancho/2, alto - my - 40, f"Cotización N° {numero}")

def draw_datos_cliente(pdf, nombre, rut, direccion, destinatarios, mx, y):
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Cliente: {nombre}")
    y -= 14
    pdf.setFont(*FONTS['datos'])
    pdf.drawString(mx, y, f"Rut: {rut}")
    y -= 14
    pdf.drawString(mx, y, f"Dirección: {direccion}")
    y -= 20
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Estimado/a: {destinatarios}")
    return y - 20

def draw_introduccion(pdf, descripcion, mx, y):
    texto = f"Ud. ha solicitado los precios de {descripcion}, a continuación aparece nuestra cotización:"
    text_obj = pdf.beginText(mx, y)
    pdf.setFont(*FONTS['introduccion'])
    for line in wrap(texto, width=115):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 20

def draw_tabla_items(pdf, items, ancho, mx, y):
    data = [["Descripción", "Cantidad", "Precio Unit", "Total Neto"]]
    for it in items:
        data.append([
            it['descripcion'],
            str(it['cantidad']),
            it['precio_unitario'],
            it['total_neto']
        ])
    table_width = ancho - 2 * mx
    colWidths = [0.4 * table_width, 0.15 * table_width, 0.2 * table_width, 0.25 * table_width]
    style = TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('ALIGN', (1, 1), (-1, -1), 'RIGHT'),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.black),
        ('FONTSIZE', (0, 0), (-1, -1), 10),
    ])
    tabla = Table(data, colWidths=colWidths, style=style)
    w, h = tabla.wrapOn(pdf, 0, 0)
    tabla.drawOn(pdf, mx, y - h)
    return y - h - 20

def draw_texto_fijo(pdf, mx, y, tipo_moneda):
    if tipo_moneda == '1':
        fijo = "Valores netos expresados en USD, conversión del dólar, observado del día de la compra +$5"
    else:
        fijo = "Valores netos expresados en CLP, debe agregar IVA"
    pdf.setFont(*FONTS['fijo'])
    for line in wrap(fijo, width=90):
        pdf.drawString(mx, y, line)
        y -= 12
    return y - 10

def draw_observaciones(pdf, observaciones, mx, y):
    if not observaciones:
        return y
    pdf.setFont(*FONTS['fijo'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(observaciones, width=90):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 20

def draw_cierre(pdf, mx, y):
    cierre = (
        "Gracias por darnos la oportunidad de ofrecerle este presupuesto. "
        "Como siempre, es para nosotros un placer hacer negocios con ustedes. "
        "Esperamos hacer realidad este pedido para su completa satisfacción."
    )
    pdf.setFont(*FONTS['cierre'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(cierre, width=115):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 30

def draw_firma(pdf, firmante, cargo, ancho, y):
    lines = ['Atentamente,', firmante]
    if cargo:
        lines.append(cargo)
    for i, line in enumerate(lines):
        if i == 0:
            pdf.setFont(*FONTS['firma_label'])
        elif i == 1:
            pdf.setFont(*FONTS['firma'])
        else:
            pdf.setFont(*FONTS['firma_cargo'])
        pdf.drawCentredString(ancho/2, y - i*14, line)
    return y - len(lines)*14 - 10

def draw_pie(pdf, contactos, ancho, my):
    pdf.setFont(*FONTS['footer'])
    footer_y = my - 10
    for idx, link in enumerate(contactos):
        y = footer_y + (len(contactos) - idx) * 10
        pdf.drawCentredString(ancho/2, y, link)
    page = pdf.getPageNumber()
    pdf.drawCentredString(ancho/2, footer_y - 20, f"{page}/{page}")

# ——— Función principal ———
def generar_pdf_cotizacion(
    datos_cotizacion,
    logo_base64,
    nombre_empresa, rut_empresa, direccion_empresa,
    telefono_empresa, email_empresa, sitio_web_empresa,
    nombre_cliente, rut_cliente, direccion_cliente,
    destinatarios,
    items=None, observaciones=None,
    cierre=None, firmante=None, cargo=None,
    contactos=None,
    ubicacion="Santiago", tipo_moneda='1'
):
    if items is None: items = []
    if contactos is None: contactos = []

    buffer = BytesIO()
    pdf = canvas.Canvas(buffer, pagesize=A4)
    ancho, alto = A4
    mx, my = 40, 40

    fecha_str = datos_cotizacion.get('fecha_cotizacion',
                                     datetime.now().strftime("%d de %B de %Y"))

    # Encabezado
    draw_encabezado(pdf, ubicacion, fecha_str, logo_base64, ancho, alto, mx, my)
    # Título
    draw_titulo(pdf, datos_cotizacion.get('numero_cotizacion', ''), ancho, alto, mx, my)

    # Datos cliente y saludo
    y = alto - my - 90
    y = draw_datos_cliente(
        pdf, nombre_cliente, rut_cliente, direccion_cliente,
        destinatarios, mx, y
    )

    # Introducción fija con descripción
    desc = datos_cotizacion.get('descripcion', '')
    y = draw_introduccion(pdf, desc, mx, y)

    # Tabla de ítems
    y = draw_tabla_items(pdf, items, ancho, mx, y)

    # Texto fijo según tipo de moneda
    y = draw_texto_fijo(pdf, mx, y, tipo_moneda)

    # Observaciones
    y = draw_observaciones(pdf, observaciones, mx, y)

    # Cierre y firma
    y = draw_cierre(pdf, mx, y)
    y = draw_firma(pdf, firmante, cargo, ancho, y)

    # Pie de página
    draw_pie(pdf, contactos, ancho, my)

    pdf.showPage()
    pdf.save()
    buffer.seek(0)
    return buffer.getvalue()

# ——— Wrapper para Jupyter ———
def generar_pdf_cotizacion_desde_model(cotizacion_id, ubicacion="Santiago"):
    from cotizaciones.models import Cotizacion
    cot = (
        Cotizacion.objects
        .select_related('empresa', 'cliente')
        .prefetch_related('items', 'solicitantes')
        .get(pk=cotizacion_id)
    )
    # Datos básicos
    datos_cotizacion = {
        'numero_cotizacion': cot.numero_cotizacion,
        'descripcion': cot.descripcion or ''
    }
    datos_cotizacion['fecha_cotizacion'] = cot.fecha_creacion.strftime("%d de %B de %Y")

    # Logo
    logo_b64 = None
    logo_field = getattr(cot.empresa, 'logo', None)
    logo_bytes = None
    if logo_field:
        if hasattr(logo_field, 'read'):
            try:
                logo_bytes = logo_field.read()
            except:
                logo_bytes = None
        elif isinstance(logo_field, str) and ',' in logo_field:
            logo_b64 = logo_field
        else:
            try:
                with open(logo_field, 'rb') as f:
                    logo_bytes = f.read()
            except:
                logo_bytes = None
    if logo_bytes:
        logo_b64 = 'data:image/png;base64,' + base64.b64encode(logo_bytes).decode()

    # Solicitantes
    destinatarios = '/'.join(str(s.usuario) for s in cot.solicitantes.all()) or ''

    # Items
    items = []
    for it in cot.items.all():
        items.append({
            'descripcion':    it.descripcion or it.nombre or '',
            'cantidad':       it.cantidad,
            'precio_unitario': f"{it.precio_unitario:.2f}",
            'total_neto':      f"{it.costo_total:.2f}"
        })

    # Contactos
    contactos = []
    if cot.empresa.telefono:   contactos.append(cot.empresa.telefono)
    if cot.empresa.email:      contactos.append(cot.empresa.email)
    if getattr(cot.empresa, 'sitio_web', None): contactos.append(cot.empresa.sitio_web)

    # Generación
    return generar_pdf_cotizacion(
        datos_cotizacion=datos_cotizacion,
        logo_base64=logo_b64,
        nombre_empresa=cot.empresa.nombre,
        rut_empresa=getattr(cot.empresa, 'rut_empresa', ''),
        direccion_empresa=getattr(cot.empresa, 'direccion_principal', ''),
        telefono_empresa=cot.empresa.telefono or '',
        email_empresa=cot.empresa.email or '',
        sitio_web_empresa=getattr(cot.empresa, 'sitio_web', ''),
        nombre_cliente=cot.cliente.nombre,
        rut_cliente=getattr(cot.cliente, 'rut_empresa', ''),
        direccion_cliente=getattr(cot.cliente, 'direccion_principal', ''),
        destinatarios=destinatarios,
        items=items,
        observaciones=cot.observaciones or '',
        cierre='',
        firmante=cot.empresa.nombre,
        cargo='',
        contactos=contactos,
        ubicacion=ubicacion,
        tipo_moneda=cot.tipo_moneda
    )

In [ ]:
%load_ext autoreload
%autoreload 2
from IPython.display import display, IFrame
pdf_bytes = generar_pdf_cotizacion_desde_model(1)
with open('cot_db.pdf', 'wb') as f:
    f.write(pdf_bytes)
display(IFrame('cot_db.pdf', width=700, height=500))

In [ ]:
cot = (
    Cotizacion.objects
    .select_related('empresa','cliente')
    .prefetch_related('items','solicitantes')
    .get(pk=1)
)

In [ ]:
cot.solicitantes.all()

In [ ]:
from io import BytesIO
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle, Paragraph
from reportlab.lib.utils import ImageReader
from reportlab.lib.styles import ParagraphStyle
from textwrap import wrap
import base64

# ——— Constantes de fuentes ———
FONTS = {
    'fecha':        ("Helvetica",        9),
    'titulo':       ("Helvetica-Bold",  14),
    'datos_label':  ("Helvetica-Bold",  10),
    'datos':        ("Helvetica",       10),
    'introduccion': ("Helvetica",       10),
    'tabla_head':   ("Helvetica-Bold",  10),
    'tabla':        ("Helvetica",       10),
    'fijo':         ("Helvetica-Oblique",9),
    'cierre':       ("Helvetica",       10),
    'firma_label':  ("Helvetica-Bold",  10),
    'firma':        ("Helvetica",       10),
    'firma_cargo':  ("Helvetica",        9),
    'footer':       ("Helvetica",        8),
}

# ——— Estilo para párrafo de introducción ———
INTRO_STYLE = ParagraphStyle(
    'intro',
    fontName='Helvetica',
    fontSize=10,
    leading=12
)

# ——— Funciones de dibujo ———
def draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my):
    pdf.setFont(*FONTS['fecha'])
    pdf.drawString(mx, alto - my + 1, f"{ubicacion}, {fecha_str}")

def draw_logo(pdf, logo_b64, ancho, alto, mx, my):
    if not logo_b64:
        return
    try:
        b64 = logo_b64.split(',', 1)[1] if ',' in logo_b64 else logo_b64
        img = ImageReader(BytesIO(base64.b64decode(b64)))
        iw, ih = img.getSize()
        w_logo = 120
        h_logo = w_logo * (ih / iw)
        pdf.drawImage(img, ancho - mx - w_logo, alto - my - h_logo,
                      width=w_logo, height=h_logo, mask='auto')
    except:
        pass

def draw_separator(pdf, mx, ancho, alto, my):
    pdf.setStrokeColor(colors.grey)
    pdf.setLineWidth(0.5)
    pdf.line(mx, alto - my - 60, ancho - mx, alto - my - 60)

def draw_encabezado(pdf, ubicacion, fecha_str, logo_b64, ancho, alto, mx, my):
    draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my)
    draw_logo(pdf, logo_b64, ancho, alto, mx, my)
    # draw_separator(pdf, mx, ancho, alto, my)

def draw_titulo(pdf, numero, ancho, alto, mx, my):
    pdf.setFont(*FONTS['titulo'])
    pdf.drawCentredString(ancho/2, alto - my - 40, f"Cotización N° {numero}")

def draw_datos_cliente(pdf, nombre, rut, direccion, destinatarios, mx, y):
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Cliente: {nombre}")
    y -= 14
    pdf.setFont(*FONTS['datos'])
    pdf.drawString(mx, y, f"Rut: {rut}")
    y -= 14
    pdf.drawString(mx, y, f"Dirección: {direccion}")
    y -= 20
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Estimado/a: {destinatarios}")
    return y - 20

# ——— Introducción con descripción en negrita ———
def draw_introduccion(pdf, descripcion, mx, y, ancho):
    intro_text = (
        f"Ud. ha solicitado los precios de <b>{descripcion}</b>, "
        "a continuación aparece nuestra cotización:"
    )
    para = Paragraph(intro_text, INTRO_STYLE)
    available_width = ancho - 2 * mx
    w, h = para.wrap(available_width, y)
    para.drawOn(pdf, mx, y - h)
    return y - h - 12

# ——— Resto de funciones sin cambios ———
def draw_tabla_items(pdf, items, ancho, mx, y):
    data = [["Descripción", "Cantidad", "Precio Unit", "Total Neto"]]
    for it in items:
        data.append([
            it['descripcion'],
            str(it['cantidad']),
            it['precio_unitario'],
            it['total_neto']
        ])
    table_width = ancho - 2 * mx
    colWidths = [0.4 * table_width, 0.15 * table_width, 0.2 * table_width, 0.25 * table_width]
    style = TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('ALIGN', (1, 1), (-1, -1), 'RIGHT'),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.black),
        ('FONTSIZE', (0, 0), (-1, -1), 10),
    ])
    tabla = Table(data, colWidths=colWidths, style=style)
    w, h = tabla.wrapOn(pdf, 0, 0)
    tabla.drawOn(pdf, mx, y - h)
    return y - h - 20

def draw_texto_fijo(pdf, mx, y, tipo_moneda):
    if tipo_moneda == '1':
        fijo = "Valores netos expresados en USD, conversión del dólar, observado del día de la compra +$5"
    else:
        fijo = "Valores netos expresados en CLP, debe agregar IVA"
    pdf.setFont(*FONTS['fijo'])
    for line in wrap(fijo, width=90):
        pdf.drawString(mx, y, line)
        y -= 12
    return y - 10

def draw_observaciones(pdf, observaciones, mx, y):
    if not observaciones:
        return y
    pdf.setFont(*FONTS['fijo'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(observaciones, width=90):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 20

def draw_cierre(pdf, mx, y):
    cierre = (
        "Gracias por darnos la oportunidad de ofrecerle este presupuesto. "
        "Como siempre, es para nosotros un placer hacer negocios con ustedes. "
        "Esperamos hacer realidad este pedido para su completa satisfacción."
    )
    pdf.setFont(*FONTS['cierre'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(cierre, width=115):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 30

def draw_firma(pdf, firmante, cargo, ancho, y):
    lines = ['Atentamente,', firmante]
    if cargo:
        lines.append(cargo)
    for i, line in enumerate(lines):
        if i == 0:
            pdf.setFont(*FONTS['firma_label'])
        elif i == 1:
            pdf.setFont(*FONTS['firma'])
        else:
            pdf.setFont(*FONTS['firma_cargo'])
        pdf.drawCentredString(ancho/2, y - i*14, line)
    return y - len(lines)*14 - 10

def draw_pie(pdf, contactos, ancho, my):
    pdf.setFont(*FONTS['footer'])
    footer_y = my - 10
    for idx, link in enumerate(contactos):
        y = footer_y + (len(contactos) - idx) * 10
        pdf.drawCentredString(ancho/2, y, link)
    page = pdf.getPageNumber()
    pdf.drawCentredString(ancho/2, footer_y - 20, f"{page}/{page}")

# ——— Función principal ———
def generar_pdf_cotizacion(
    datos_cotizacion,
    logo_base64,
    nombre_empresa, rut_empresa, direccion_empresa,
    telefono_empresa, email_empresa, sitio_web_empresa,
    nombre_cliente, rut_cliente, direccion_cliente,
    destinatarios,
    items=None, observaciones=None,
    cierre=None, firmante=None, cargo=None,
    contactos=None,
    ubicacion="Santiago", tipo_moneda='1'
):
    if items is None: items = []
    if contactos is None: contactos = []

    buffer = BytesIO()
    pdf = canvas.Canvas(buffer, pagesize=A4)
    ancho, alto = A4
    mx, my = 40, 40

    fecha_str = datos_cotizacion.get('fecha_cotizacion',
                                     datetime.now().strftime("%d de %B de %Y"))

    draw_encabezado(pdf, ubicacion, fecha_str, logo_base64, ancho, alto, mx, my)
    draw_titulo(pdf, datos_cotizacion.get('numero_cotizacion',''), ancho, alto, mx, my)
    y = alto - my - 90
    y = draw_datos_cliente(pdf, nombre_cliente, rut_cliente, direccion_cliente, destinatarios, mx, y)

    # Introducción con negrita para descripción
    y = draw_introduccion(pdf, datos_cotizacion.get('descripcion',''), mx, y, ancho)

    y = draw_tabla_items(pdf, items, ancho, mx, y)
    y = draw_texto_fijo(pdf, mx, y, tipo_moneda)
    y = draw_observaciones(pdf, observaciones, mx, y)
    y = draw_cierre(pdf, mx, y)
    y = draw_firma(pdf, firmante, cargo, ancho, y)
    draw_pie(pdf, contactos, ancho, my)

    pdf.showPage()
    pdf.save()
    buffer.seek(0)
    return buffer.getvalue()

# ——— Wrapper para Jupyter ———
def generar_pdf_cotizacion_desde_model(cotizacion_id, ubicacion="Santiago"):
    from cotizaciones.models import Cotizacion
    cot = (
        Cotizacion.objects
        .select_related('empresa','cliente')
        .prefetch_related('items','solicitantes')
        .get(pk=cotizacion_id)
    )
    datos_cotizacion = {
        'numero_cotizacion': cot.numero_cotizacion,
        'descripcion': cot.descripcion or ''
    }
    datos_cotizacion['fecha_cotizacion'] = cot.fecha_creacion.strftime("%d de %B de %Y")

    logo_b64 = None
    logo_field = getattr(cot.empresa,'logo',None)
    logo_bytes=None
    if logo_field and hasattr(logo_field,'read'):
        try: logo_bytes = logo_field.read()
        except: logo_bytes=None
    elif isinstance(logo_field,str) and ',' in logo_field:
        logo_b64 = logo_field
    elif isinstance(logo_field,str):
        try:
            with open(logo_field,'rb') as f: logo_bytes=f.read()
        except: logo_bytes=None
    if logo_bytes:
        logo_b64='data:image/png;base64,'+base64.b64encode(logo_bytes).decode()

    destinatarios='/'.join(str(s.usuario) for s in cot.solicitantes.all()) or ''
    items=[]
    for it in cot.items.all():
        items.append({
            'descripcion': it.descripcion or it.nombre or '',
            'cantidad': it.cantidad,
            'precio_unitario': f"{it.precio_unitario:.2f}",
            'total_neto': f"{it.costo_total:.2f}"
        })
    contactos=[]
    if cot.empresa.telefono: contactos.append(cot.empresa.telefono)
    if cot.empresa.email:    contactos.append(cot.empresa.email)
    if getattr(cot.empresa,'sitio_web',None): contactos.append(cot.empresa.sitio_web)

    return generar_pdf_cotizacion(
        datos_cotizacion, logo_b64,
        cot.empresa.nombre, getattr(cot.empresa,'rut_empresa',''), getattr(cot.empresa,'direccion_principal',''),
        cot.empresa.telefono or '', cot.empresa.email or '', getattr(cot.empresa,'sitio_web',''),
        cot.cliente.nombre, getattr(cot.cliente,'rut_empresa',''), getattr(cot.cliente,'direccion_principal',''),
        destinatarios, items, cot.observaciones or '', '', cot.empresa.nombre, '', contactos, ubicacion, cot.tipo_moneda
    )

In [ ]:
%load_ext autoreload
%autoreload 2
from IPython.display import display, IFrame
pdf_bytes = generar_pdf_cotizacion_desde_model(1)
with open('cot_db.pdf', 'wb') as f:
    f.write(pdf_bytes)
display(IFrame('cot_db.pdf', width=700, height=500))

In [ ]:
from io import BytesIO
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle, Paragraph
from reportlab.lib.utils import ImageReader
from reportlab.lib.styles import ParagraphStyle
from textwrap import wrap
import base64

# ——— Constantes de fuentes ———
FONTS = {
    'fecha':        ("Helvetica",        9),
    'titulo':       ("Helvetica-Bold",  14),
    'datos_label':  ("Helvetica-Bold",  10),
    'datos':        ("Helvetica",       10),
    'introduccion': ("Helvetica",       10),
    'tabla_head':   ("Helvetica-Bold",  10),
    'tabla':        ("Helvetica",       10),
    'fijo':         ("Helvetica-Oblique",9),
    'cierre':       ("Helvetica",       10),
    'firma_label':  ("Helvetica-Bold",  10),
    'firma':        ("Helvetica",       10),
    'firma_cargo':  ("Helvetica",        9),
    'footer':       ("Helvetica",        8),
}

# ——— Estilo para párrafo de introducción ———
INTRO_STYLE = ParagraphStyle(
    'intro',
    fontName='Helvetica',
    fontSize=10,
    leading=12
)

# ——— Funciones de dibujo ———
def draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my):
    pdf.setFont(*FONTS['fecha'])
    pdf.drawString(mx, alto - my + 1, f"{ubicacion}, {fecha_str}")

def draw_logo(pdf, logo_b64, ancho, alto, mx, my):
    if not logo_b64:
        return
    try:
        b64 = logo_b64.split(',', 1)[1] if ',' in logo_b64 else logo_b64
        img = ImageReader(BytesIO(base64.b64decode(b64)))
        iw, ih = img.getSize()
        w_logo = 120
        h_logo = w_logo * (ih / iw)
        pdf.drawImage(img, ancho - mx - w_logo, alto - my - h_logo,
                      width=w_logo, height=h_logo, mask='auto')
    except:
        pass

def draw_separator(pdf, mx, ancho, alto, my):
    pdf.setStrokeColor(colors.grey)
    pdf.setLineWidth(0.5)
    pdf.line(mx, alto - my - 60, ancho - mx, alto - my - 60)

def draw_encabezado(pdf, ubicacion, fecha_str, logo_b64, ancho, alto, mx, my):
    draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my)
    draw_logo(pdf, logo_b64, ancho, alto, mx, my)
    # draw_separator(pdf, mx, ancho, alto, my)

def draw_titulo(pdf, numero, ancho, alto, mx, my):
    pdf.setFont(*FONTS['titulo'])
    pdf.drawCentredString(ancho/2, alto - my - 40, f"Cotización N° {numero}")

def draw_datos_cliente(pdf, nombre, rut, direccion, destinatarios, mx, y):
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Cliente: {nombre}")
    y -= 14
    pdf.setFont(*FONTS['datos'])
    pdf.drawString(mx, y, f"Rut: {rut}")
    y -= 14
    pdf.drawString(mx, y, f"Dirección: {direccion}")
    y -= 20
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Estimado/a: {destinatarios}")
    return y - 20

# ——— Introducción con descripción en negrita ———
def draw_introduccion(pdf, descripcion, mx, y, ancho):
    intro_text = (
        f"Ud. ha solicitado los precios de <b>{descripcion}</b>, "
        "a continuación aparece nuestra cotización:"
    )
    para = Paragraph(intro_text, INTRO_STYLE)
    available_width = ancho - 2 * mx
    w, h = para.wrap(available_width, y)
    para.drawOn(pdf, mx, y - h)
    return y - h - 12

# ——— Resto de funciones sin cambios ———
def draw_tabla_items(pdf, items, ancho, mx, y):
    data = [["Descripción", "Cantidad", "Precio Unit", "Total Neto"]]
    for it in items:
        data.append([
            it['descripcion'],
            str(it['cantidad']),
            it['precio_unitario'],
            it['total_neto']
        ])
    table_width = ancho - 2 * mx
    colWidths = [0.4 * table_width, 0.15 * table_width, 0.2 * table_width, 0.25 * table_width]
    style = TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('ALIGN', (1, 1), (-1, -1), 'RIGHT'),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.black),
        ('FONTSIZE', (0, 0), (-1, -1), 10),
    ])
    tabla = Table(data, colWidths=colWidths, style=style)
    w, h = tabla.wrapOn(pdf, 0, 0)
    tabla.drawOn(pdf, mx, y - h)
    return y - h - 20

def draw_texto_fijo(pdf, mx, y, tipo_moneda):
    if tipo_moneda == '1':
        fijo = "Valores netos expresados en USD, conversión del dólar, observado del día de la compra +$5"
    else:
        fijo = "Valores netos expresados en CLP, debe agregar IVA"
    pdf.setFont(*FONTS['fijo'])
    for line in wrap(fijo, width=90):
        pdf.drawString(mx, y, line)
        y -= 12
    return y - 10

def draw_observaciones(pdf, observaciones, mx, y):
    if not observaciones:
        return y
    pdf.setFont(*FONTS['fijo'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(observaciones, width=90):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 20

def draw_cierre(pdf, mx, y):
    cierre = (
        "Gracias por darnos la oportunidad de ofrecerle este presupuesto. "
        "Como siempre, es para nosotros un placer hacer negocios con ustedes. "
        "Esperamos hacer realidad este pedido para su completa satisfacción."
    )
    pdf.setFont(*FONTS['cierre'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(cierre, width=115):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 30

def draw_firma(pdf, firmante, cargo, ancho, y):
    lines = ['Atentamente,', firmante]
    if cargo:
        lines.append(cargo)
    for i, line in enumerate(lines):
        if i == 0:
            pdf.setFont(*FONTS['firma_label'])
        elif i == 1:
            pdf.setFont(*FONTS['firma'])
        else:
            pdf.setFont(*FONTS['firma_cargo'])
        pdf.drawCentredString(ancho/2, y - i*14, line)
    return y - len(lines)*14 - 10

def draw_pie(pdf, _, ancho, my):
    # Pie de página fijo con hyperlink y paginación
    pdf.setFont(*FONTS['footer'])
    footer_y = my - 30
    lines = [
        "Snabb IT | Una empresa del Grupo A&G",
        "Av. José Miguel Carrera N° 3840 Of- 808, San Miguel - Fono: (56-2) 27596140",
        "email: lrojas@snabb-it.cl administracion@aygasociados.cl",
        "Visítenos en https://snabbit.cl"
    ]
    for i, line in enumerate(lines):
        y_line = footer_y + i * 12
        pdf.drawCentredString(ancho/2, y_line, line)
        # Agregar hyperlink en la URL si existe
        if "https://" in line:
            url = line.split()[-1]
            text_width = pdf.stringWidth(line, FONTS['footer'][0], FONTS['footer'][1])
            x_start = (ancho - text_width) / 2
            x_end = x_start + text_width
            y_bottom = y_line
            y_top = y_bottom + FONTS['footer'][1]
            pdf.linkURL(url, (x_start, y_bottom, x_end, y_top), relative=0)
    # Paginación (x de x)
    page = pdf.getPageNumber()
    total = page  # como no hay contador global, se muestra x/x
    pdf.drawCentredString(ancho/2, footer_y - 12, f"{page}/{total}")

# ——— Función principal ———# ——— Función principal ———
def generar_pdf_cotizacion(
    datos_cotizacion,
    logo_base64,
    nombre_empresa, rut_empresa, direccion_empresa,
    telefono_empresa, email_empresa, sitio_web_empresa,
    nombre_cliente, rut_cliente, direccion_cliente,
    destinatarios,
    items=None, observaciones=None,
    cierre=None, firmante=None, cargo=None,
    contactos=None,
    ubicacion="Santiago", tipo_moneda='1'
):
    if items is None: items = []
    if contactos is None: contactos = []

    buffer = BytesIO()
    pdf = canvas.Canvas(buffer, pagesize=A4)
    ancho, alto = A4
    mx, my = 40, 40

    fecha_str = datos_cotizacion.get('fecha_cotizacion',
                                     datetime.now().strftime("%d de %B de %Y"))

    draw_encabezado(pdf, ubicacion, fecha_str, logo_base64, ancho, alto, mx, my)
    draw_titulo(pdf, datos_cotizacion.get('numero_cotizacion',''), ancho, alto, mx, my)
    y = alto - my - 90
    y = draw_datos_cliente(pdf, nombre_cliente, rut_cliente, direccion_cliente, destinatarios, mx, y)

    # Introducción con negrita para descripción
    y = draw_introduccion(pdf, datos_cotizacion.get('descripcion',''), mx, y, ancho)

    y = draw_tabla_items(pdf, items, ancho, mx, y)
    y = draw_texto_fijo(pdf, mx, y, tipo_moneda)
    y = draw_observaciones(pdf, observaciones, mx, y)
    y = draw_cierre(pdf, mx, y)
    y = draw_firma(pdf, firmante, cargo, ancho, y)
    draw_pie(pdf, contactos, ancho, my)

    pdf.showPage()
    pdf.save()
    buffer.seek(0)
    return buffer.getvalue()

# ——— Wrapper para Jupyter ———
def generar_pdf_cotizacion_desde_model(cotizacion_id, ubicacion="Santiago"):
    from cotizaciones.models import Cotizacion
    cot = (
        Cotizacion.objects
        .select_related('empresa','cliente')
        .prefetch_related('items','solicitantes')
        .get(pk=cotizacion_id)
    )
    datos_cotizacion = {
        'numero_cotizacion': cot.numero_cotizacion,
        'descripcion': cot.descripcion or ''
    }
    datos_cotizacion['fecha_cotizacion'] = cot.fecha_creacion.strftime("%d de %B de %Y")

    logo_b64 = None
    logo_field = getattr(cot.empresa,'logo',None)
    logo_bytes=None
    if logo_field and hasattr(logo_field,'read'):
        try: logo_bytes = logo_field.read()
        except: logo_bytes=None
    elif isinstance(logo_field,str) and ',' in logo_field:
        logo_b64 = logo_field
    elif isinstance(logo_field,str):
        try:
            with open(logo_field,'rb') as f: logo_bytes=f.read()
        except: logo_bytes=None
    if logo_bytes:
        logo_b64='data:image/png;base64,'+base64.b64encode(logo_bytes).decode()

    destinatarios='/'.join(str(s) for s in cot.solicitantes.all()) or ''
    items=[]
    for it in cot.items.all():
        items.append({
            'descripcion': it.descripcion or it.nombre or '',
            'cantidad': it.cantidad,
            'precio_unitario': f"{it.precio_unitario:.2f}",
            'total_neto': f"{it.costo_total:.2f}"
        })
    contactos=[]
    if cot.empresa.telefono: contactos.append(cot.empresa.telefono)
    if cot.empresa.email:    contactos.append(cot.empresa.email)
    if getattr(cot.empresa,'sitio_web',None): contactos.append(cot.empresa.sitio_web)

    return generar_pdf_cotizacion(
        datos_cotizacion, logo_b64,
        cot.empresa.nombre, getattr(cot.empresa,'rut_empresa',''), getattr(cot.empresa,'direccion_principal',''),
        cot.empresa.telefono or '', cot.empresa.email or '', getattr(cot.empresa,'sitio_web',''),
        cot.cliente.nombre, getattr(cot.cliente,'rut_empresa',''), getattr(cot.cliente,'direccion_principal',''),
        destinatarios, items, cot.observaciones or '', '', cot.empresa.nombre, '', contactos, ubicacion, cot.tipo_moneda
    )


In [ ]:
%load_ext autoreload
%autoreload 2
from IPython.display import display, IFrame
pdf_bytes = generar_pdf_cotizacion_desde_model(1)
with open('cot_db.pdf', 'wb') as f:
    f.write(pdf_bytes)
display(IFrame('cot_db.pdf', width=700, height=500))

In [ ]:
from io import BytesIO
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle, Paragraph
from reportlab.lib.utils import ImageReader
from reportlab.lib.styles import ParagraphStyle
from textwrap import wrap
import base64

# ——— Constantes de fuentes ———
FONTS = {
    'fecha':        ("Helvetica",        9),
    'titulo':       ("Helvetica-Bold",  14),
    'datos_label':  ("Helvetica-Bold",  10),
    'datos':        ("Helvetica",       10),
    'introduccion': ("Helvetica",       10),
    'tabla_head':   ("Helvetica-Bold",  10),
    'tabla':        ("Helvetica",       10),
    'fijo':         ("Helvetica-Oblique",9),
    'cierre':       ("Helvetica",       10),
    'firma_label':  ("Helvetica-Bold",  10),
    'firma':        ("Helvetica",       10),
    'firma_cargo':  ("Helvetica",        9),
    'footer':       ("Helvetica",        8),
}

# ——— Estilo para párrafo de introducción ———
INTRO_STYLE = ParagraphStyle(
    'intro',
    fontName='Helvetica',
    fontSize=10,
    leading=12
)

# ——— Funciones de dibujo ———
def draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my):
    pdf.setFont(*FONTS['fecha'])
    pdf.drawString(mx, alto - my + 1, f"{ubicacion}, {fecha_str}")

def draw_logo(pdf, logo_b64, ancho, alto, mx, my):
    if not logo_b64:
        return
    try:
        b64 = logo_b64.split(',', 1)[1] if ',' in logo_b64 else logo_b64
        img = ImageReader(BytesIO(base64.b64decode(b64)))
        iw, ih = img.getSize()
        w_logo = 120
        h_logo = w_logo * (ih / iw)
        pdf.drawImage(img, ancho - mx - w_logo, alto - my - h_logo,
                      width=w_logo, height=h_logo, mask='auto')
    except:
        pass

def draw_encabezado(pdf, ubicacion, fecha_str, logo_b64, ancho, alto, mx, my):
    draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my)
    draw_logo(pdf, logo_b64, ancho, alto, mx, my)

def draw_titulo(pdf, numero, ancho, alto, mx, my):
    pdf.setFont(*FONTS['titulo'])
    pdf.drawCentredString(ancho/2, alto - my - 40, f"Cotización N° {numero}")

def draw_datos_cliente(pdf, nombre, rut, direccion, destinatarios, mx, y):
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Cliente: {nombre}")
    y -= 14
    pdf.setFont(*FONTS['datos'])
    pdf.drawString(mx, y, f"Rut: {rut}")
    y -= 14
    pdf.drawString(mx, y, f"Dirección: {direccion}")
    y -= 20
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Estimado/a: {destinatarios}")
    return y - 20

# ——— Introducción con descripción en negrita ———
def draw_introduccion(pdf, descripcion, mx, y, ancho):
    intro_text = (
        f"Ud. ha solicitado los precios de <b>{descripcion}</b>, "
        "a continuación aparece nuestra cotización:"
    )
    para = Paragraph(intro_text, INTRO_STYLE)
    available_width = ancho - 2 * mx
    w, h = para.wrap(available_width, y)
    para.drawOn(pdf, mx, y - h)
    return y - h - 12

# ——— Tabla de ítems ———
def draw_tabla_items(pdf, items, ancho, mx, y):
    data = [["Descripción", "Cantidad", "Precio Unit", "Total Neto"]]
    for it in items:
        data.append([
            it['descripcion'],
            str(it['cantidad']),
            it['precio_unitario'],
            it['total_neto']
        ])
    table_width = ancho - 2 * mx
    colWidths = [0.4 * table_width, 0.15 * table_width, 0.2 * table_width, 0.25 * table_width]
    style = TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('FONTNAME',   (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('ALIGN',      (1, 1), (-1, -1), 'RIGHT'),
        ('GRID',       (0, 0), (-1, -1), 0.5, colors.black),
        ('FONTSIZE',   (0, 0), (-1, -1), 10),
    ])
    tabla = Table(data, colWidths=colWidths, style=style)
    w, h = tabla.wrapOn(pdf, 0, 0)
    tabla.drawOn(pdf, mx, y - h)
    return y - h - 20

# ——— Texto fijo según moneda ———
def draw_texto_fijo(pdf, mx, y, tipo_moneda):
    if tipo_moneda == '1':
        fijo = "Valores netos expresados en USD, conversión del dólar, observado del día de la compra +$5"
    else:
        fijo = "Valores netos expresados en CLP, debe agregar IVA"
    pdf.setFont(*FONTS['fijo'])
    for line in wrap(fijo, width=90):
        pdf.drawString(mx, y, line)
        y -= 12
    return y - 10

# ——— Observaciones ———
def draw_observaciones(pdf, observaciones, mx, y):
    if not observaciones:
        return y
    pdf.setFont(*FONTS['fijo'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(observaciones, width=90):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 20

# ——— Cierre y firma ———
def draw_cierre(pdf, mx, y):
    cierre = (
        "Gracias por darnos la oportunidad de ofrecerle este presupuesto. "
        "Como siempre, es para nosotros un placer hacer negocios con ustedes. "
        "Esperamos hacer realidad este pedido para su completa satisfacción."
    )
    pdf.setFont(*FONTS['cierre'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(cierre, width=115):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 30

# ——— Firma ———
def draw_firma(pdf, firmante, cargo, ancho, y):
    lines = ['Atentamente,', firmante]
    if cargo:
        lines.append(cargo)
    for i, line in enumerate(lines):
        if i == 0:
            pdf.setFont(*FONTS['firma_label'])
        elif i == 1:
            pdf.setFont(*FONTS['firma'])
        else:
            pdf.setFont(*FONTS['firma_cargo'])
        pdf.drawCentredString(ancho/2, y - i*14, line)
    return y - len(lines)*14 - 10

# ——— Pie de página con hyperlink y paginación inferior derecha ———
def draw_pie(pdf, contactos, ancho, my, mx=40):
    pdf.setFont(*FONTS['footer'])
    footer_y = my - 30
    # Texto centrado
    lines = [
        "Snabb IT | Una empresa del Grupo A&G",
        "Av. José Miguel Carrera N° 3840 Of- 808, San Miguel - Fono: (56-2) 27596140",
        "email: lrojas@snabb-it.cl administracion@aygasociados.cl",
        "Visítenos en https://snabbit.cl"
    ]
    for i, line in enumerate(lines):
        y_line = footer_y + i * 12
        pdf.drawCentredString(ancho/2, y_line, line)
        if "https://" in line:
            url = line.split()[-1]
            text_width = pdf.stringWidth(line, FONTS['footer'][0], FONTS['footer'][1])
            x_start = (ancho - text_width) / 2
            x_end = x_start + text_width
            y_bottom = y_line - 2
            y_top = y_bottom + FONTS['footer'][1] + 2
            pdf.linkURL(url, (x_start, y_bottom, x_end, y_top), relative=0)
    # Paginación en esquina inferior derecha
    page = pdf.getPageNumber()
    total = page
    pag_text = f"{page}/{total}"
    pdf.drawRightString(ancho - mx, footer_y - 2, pag_text)

# ——— Función principal ———
def generar_pdf_cotizacion(
    datos_cotizacion,
    logo_base64,
    nombre_empresa, rut_empresa, direccion_empresa,
    telefono_empresa, email_empresa, sitio_web_empresa,
    nombre_cliente, rut_cliente, direccion_cliente,
    destinatarios,
    items=None, observaciones=None,
    cierre=None, firmante=None, cargo=None,
    contactos=None,
    ubicacion="Santiago", tipo_moneda='1'
):
    if items is None: items = []
    if contactos is None: contactos = []

    buffer = BytesIO()
    pdf = canvas.Canvas(buffer, pagesize=A4)
    ancho, alto = A4
    mx, my = 40, 40

    fecha_str = datos_cotizacion.get('fecha_cotizacion', datetime.now().strftime("%d de %B de %Y"))

    draw_encabezado(pdf, ubicacion, fecha_str, logo_base64, ancho, alto, mx, my)
    draw_titulo(pdf, datos_cotizacion.get('numero_cotizacion',''), ancho, alto, mx, my)
    y = alto - my - 90
    y = draw_datos_cliente(pdf, nombre_cliente, rut_cliente, direccion_cliente, destinatarios, mx, y)
    y = draw_introduccion(pdf, datos_cotizacion.get('descripcion',''), mx, y, ancho)
    y = draw_tabla_items(pdf, items, ancho, mx, y)
    y = draw_texto_fijo(pdf, mx, y, tipo_moneda)
    y = draw_observaciones(pdf, observaciones, mx, y)
    y = draw_cierre(pdf, mx, y)
    y = draw_firma(pdf, firmante, cargo, ancho, y)
    draw_pie(pdf, contactos, ancho, my)

    pdf.showPage()
    pdf.save()
    buffer.seek(0)
    return buffer.getvalue()

# ——— Wrapper para Jupyter ———
def generar_pdf_cotizacion_desde_model(cotizacion_id, ubicacion="Santiago"):
    from cotizaciones.models import Cotizacion
    cot = (
        Cotizacion.objects
        .select_related('empresa','cliente')
        .prefetch_related('items','solicitantes')
        .get(pk=cotizacion_id)
    )
    datos_cotizacion = {
        'numero_cotizacion': cot.numero_cotizacion,
        'descripcion': cot.descripcion or ''
    }
    datos_cotizacion['fecha_cotizacion'] = cot.fecha_creacion.strftime("%d de %B de %Y")

    logo_b64 = None
    logo_field = getattr(cot.empresa,'logo',None)
    logo_bytes=None
    if logo_field and hasattr(logo_field,'read'):
        try: logo_bytes = logo_field.read()
        except: logo_bytes=None
    elif isinstance(logo_field,str) and ',' in logo_field:
        logo_b64 = logo_field
    elif isinstance(logo_field,str):
        try:
            with open(logo_field,'rb') as f: logo_bytes=f.read()
        except: logo_bytes=None
    if logo_bytes:
        logo_b64='data:image/png;base64,'+base64.b64encode(logo_bytes).decode()

    destinatarios = '/'.join(str(s.usuario) for s in cot.solicitantes.all()) or ''
    items=[]
    for it in cot.items.all():
        items.append({
            'descripcion':   it.descripcion or it.nombre or '',
            'cantidad':      it.cantidad,
            'precio_unitario':f"{it.precio_unitario:.2f}",
            'total_neto':     f"{it.costo_total:.2f}"
        })
    contactos=[]
    if cot.empresa.telefono:   contactos.append(cot.empresa.telefono)
    if cot.empresa.email:      contactos.append(cot.empresa.email)
    if getattr(cot.empresa,'sitio_web',None): contactos.append(cot.empresa.sitio_web)

    return generar_pdf_cotizacion(
        datos_cotizacion, logo_b64,
        cot.empresa.nombre, getattr(cot.empresa,'rut_empresa',''), cot.empresa.direccion_principal,
        cot.empresa.telefono or '', cot.empresa.email or '', getattr(cot.empresa,'sitio_web',''),
        cot.cliente.nombre, getattr(cot.cliente,'rut_empresa',''), cot.cliente.direccion_principal,
        destinatarios, items, cot.observaciones or '', '', cot.empresa.nombre, '', contactos, ubicacion, cot.tipo_moneda
    )

In [ ]:
%load_ext autoreload
%autoreload 2
from IPython.display import display, IFrame
pdf_bytes = generar_pdf_cotizacion_desde_model(1)
with open('cot_db.pdf', 'wb') as f:
    f.write(pdf_bytes)
display(IFrame('cot_db.pdf', width=1024, height=768))

In [85]:
from io import BytesIO
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle, Paragraph
from reportlab.lib.utils import ImageReader
from reportlab.lib.styles import ParagraphStyle
from textwrap import wrap
import base64

# ——— Constantes de fuentes ———
FONTS = {
    'fecha':        ("Helvetica",        9),
    'titulo':       ("Helvetica-Bold",  14),
    'datos_label':  ("Helvetica-Bold",  10),
    'datos':        ("Helvetica",       10),
    'introduccion': ("Helvetica",       10),
    'tabla_head':   ("Helvetica-Bold",  10),
    'tabla':        ("Helvetica",       10),
    'fijo':         ("Helvetica-Oblique",9),
    'cierre':       ("Helvetica",       10),
    'firma_label':  ("Helvetica-Bold",  10),
    'firma':        ("Helvetica",       10),
    'firma_cargo':  ("Helvetica",        9),
    'footer':       ("Helvetica",        8),
}

# ——— Estilo para párrafo de introducción ———
INTRO_STYLE = ParagraphStyle(
    'intro',
    fontName='Helvetica',
    fontSize=10,
    leading=12
)

# ——— Estilo para pie de página ———
FOOTER_STYLE = ParagraphStyle(
    'footer',
    fontName='Helvetica',
    fontSize=8,
    leading=10,
    alignment=1  # centered
)

# ——— Funciones de dibujo ———
def draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my):
    pdf.setFont(*FONTS['fecha'])
    pdf.drawString(mx, alto - my + 1, f"{ubicacion}, {fecha_str}")

def draw_logo(pdf, logo_b64, ancho, alto, mx, my):
    if not logo_b64:
        return
    try:
        b64 = logo_b64.split(',', 1)[1] if ',' in logo_b64 else logo_b64
        img = ImageReader(BytesIO(base64.b64decode(b64)))
        iw, ih = img.getSize()
        w_logo = 120
        h_logo = w_logo * (ih / iw)
        pdf.drawImage(img, ancho - mx - w_logo, alto - my - h_logo,
                      width=w_logo, height=h_logo, mask='auto')
    except:
        pass

def draw_encabezado(pdf, ubicacion, fecha_str, logo_b64, ancho, alto, mx, my):
    draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my)
    draw_logo(pdf, logo_b64, ancho, alto, mx, my)

def draw_titulo(pdf, numero, ancho, alto, mx, my):
    pdf.setFont(*FONTS['titulo'])
    pdf.drawCentredString(ancho/2, alto - my - 40, f"Cotización N° {numero}")

def draw_datos_cliente(pdf, nombre, rut, direccion, destinatarios, mx, y):
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Cliente: {nombre}")
    y -= 14
    pdf.setFont(*FONTS['datos'])
    pdf.drawString(mx, y, f"Rut: {rut}")
    y -= 14
    pdf.drawString(mx, y, f"Dirección: {direccion}")
    y -= 20
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Estimado/a: {destinatarios}")
    return y - 20

# ——— Introducción con descripción en negrita ———
def draw_introduccion(pdf, descripcion, mx, y, ancho):
    intro_text = (
        f"Ud. ha solicitado los precios de <b>{descripcion}</b>, "
        "a continuación aparece nuestra cotización:"
    )
    para = Paragraph(intro_text, INTRO_STYLE)
    available_width = ancho - 2 * mx
    w, h = para.wrap(available_width, y)
    para.drawOn(pdf, mx, y - h)
    return y - h - 12

# ——— Tabla de ítems ———
def draw_tabla_items(pdf, items, ancho, mx, y):
    data = [["Descripción", "Cantidad", "Precio Unit", "Total Neto"]]
    for it in items:
        data.append([
            it['descripcion'],
            str(it['cantidad']),
            it['precio_unitario'],
            it['total_neto']
        ])
    table_width = ancho - 2 * mx
    colWidths = [0.4 * table_width, 0.15 * table_width, 0.2 * table_width, 0.25 * table_width]
    style = TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('FONTNAME',   (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('ALIGN',      (1, 1), (-1, -1), 'RIGHT'),
        ('GRID',       (0, 0), (-1, -1), 0.5, colors.black),
        ('FONTSIZE',   (0, 0), (-1, -1), 10),
    ])
    tabla = Table(data, colWidths=colWidths, style=style)
    w, h = tabla.wrapOn(pdf, 0, 0)
    tabla.drawOn(pdf, mx, y - h)
    return y - h - 20

# ——— Texto fijo después de tabla ———
#def draw_texto_fijo(pdf, mx, y, tipo_moneda):
#    texto = str()
#    if tipo_moneda == '1':
#        texto = "Valores netos expresados en USD, conversión del dólar, observado del día de la compra +$5"
#    elif tipo_moneda == '2':
#        texto = "Valores netos expresados en CLP, debe agregar IVA"
#    else:
#         texto = "Valores netos expresados en UF, debe agregar IVA"
#    fijo = (texto)
#    pdf.setFont(*FONTS['fijo'])
#    for line in wrap(fijo, width=90):
#        pdf.drawString(mx, y, line)
#        y -= 12
#    return y - 10

def textotipomoneda(tipo_moneda):
    if tipo_moneda == '1':
        return "Valores netos expresados en USD, conversión del dólar, observado del día de la compra +$5"
    elif tipo_moneda == '2':
        return "Valores netos expresados en CLP, debe agregar IVA"
    elif tipo_moneda == '3':
        return "Valores netos expresados en UF, debe agregar IVA"

    
def draw_texto_fijo(pdf, mx, y, tipo_moneda):
    texto = textotipomoneda(tipo_moneda)    
    pdf.setFont(*FONTS['fijo'])
    for line in wrap(texto, width=90):
        pdf.drawString(mx, y, line)
        y -= 12
    return y - 10

# ——— Observaciones ———
def draw_observaciones(pdf, observaciones, mx, y):
    if not observaciones:
        return y
    pdf.setFont(*FONTS['fijo'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(observaciones, width=90):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 20

# ——— Cierre y firma ———
def draw_cierre(pdf, mx, y):
    cierre = (
        "Gracias por darnos la oportunidad de ofrecerle este presupuesto. "
        "Como siempre, es para nosotros un placer hacer negocios con ustedes. "
        "Esperamos hacer realidad este pedido para su completa satisfacción."
    )
    pdf.setFont(*FONTS['cierre'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(cierre, width=115):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 30

# ——— Firma ———
#def draw_firma(pdf, firmante, cargo, ancho, y):
#    lines = ['Atentamente,', firmante]
#    if cargo:
#        lines.append(cargo)
#    for i, line in enumerate(lines):
#        font = FONTS['firma_label'] if i == 0 else FONTS['firma'] if i == 1 else FONTS['firma_cargo']
#        pdf.setFont(*font)
#        pdf.drawCentredString(ancho/2, y - i*14, line)
#    return y - len(lines)*14 - 10

def draw_firma(pdf, firmante, cargo, cargo2, ancho, y):
    lines = ['Atentamente,', firmante]
    if cargo:
        lines.append(cargo)
    if cargo2:
        lines.append(cargo2)
    for i, line in enumerate(lines):
        if i == 0:
            pdf.setFont(*FONTS['firma_label'])
        elif i == 1:
            pdf.setFont(*FONTS['firma'])
        else:
            pdf.setFont(*FONTS['firma_cargo'])
        pdf.drawCentredString(ancho/2, y - i*14, line)
    return y - len(lines)*14 - 10

# ——— Pie de página ———
def draw_footer(pdf, ancho, mx, my):
    # Texto con negrita al inicio usando Paragraph
    #footer_text = (
    #    "<b>Snabb IT</b> | Una empresa del Grupo A&amp;G<br/>"
    #    "Av. José Miguel Carrera N° 3840 Of- 808, San Miguel - Fono: (56-2) 27596140<br/>"
    #    "email: lrojas@snabb-it.cl administracion@aygasociados.cl<br/>"
    #    "Visítenos en <a href='https://snabbit.cl'>https://snabbit.cl</a>"
    #)
    footer_text = (
        "<b>Snabb IT | Asesores Tecnológicos</b><br/>"
        "<a href='https://maps.app.goo.gl/R1pAm1ANqs5eEjvSA'>Gran Av. José Miguel Carrera N° 3840 Of - 808, San Miguel</a> - Fono: <a href='tel=+56227596140'>(56-2) 27596140</a><br/>"
        "Visítenos en <a href='https://snabbit.cl' target='_blank'>https://snabbit.cl</a>"
    )
    para = Paragraph(footer_text, FOOTER_STYLE)
    available_width = ancho - 2 * mx
    w, h = para.wrap(available_width, my)
    para.drawOn(pdf, mx, my - h - 5)

# ——— Paginación ———
def draw_paginacion(pdf, ancho, mx, my):
    page = pdf.getPageNumber()
    total = page  # si no hay contador global, muestra x/x
    pdf.setFont(*FONTS['footer'])
    # Esquina inferior derecha
    y = 15
    pdf.drawRightString(ancho - mx, y, f"{page}/{total}")

# ——— Función principal ———
def generar_pdf_cotizacion(
    datos_cotizacion,
    logo_base64,
    nombre_empresa, rut_empresa, direccion_empresa,
    telefono_empresa, email_empresa, sitio_web_empresa,
    nombre_cliente, rut_cliente, direccion_cliente,
    destinatarios,
    items=None, observaciones=None,
    cierre=None, firmante=None, cargo=None, cargo2=None,
    contactos="+56982983855",
    ubicacion="Santiago", tipo_moneda='1'):
    if items is None: items = []
    buffer = BytesIO()
    pdf = canvas.Canvas(buffer, pagesize=A4)
    ancho, alto = A4
    mx, my = 40, 40

    fecha_str = datos_cotizacion.get('fecha_cotizacion',
                                     datetime.now().strftime("%d de %B de %Y"))

    # Encabezado, cuerpo y pie
    draw_encabezado(pdf, ubicacion, fecha_str, logo_base64, ancho, alto, mx, my)
    draw_titulo(pdf, datos_cotizacion.get('numero_cotizacion',''), ancho, alto, mx, my)
    y = alto - my - 90
    y = draw_datos_cliente(pdf, nombre_cliente, rut_cliente, direccion_cliente, destinatarios, mx, y)
    y = draw_introduccion(pdf, datos_cotizacion.get('descripcion',''), mx, y, ancho)
    y = draw_tabla_items(pdf, items, ancho, mx, y)
    y = draw_texto_fijo(pdf, mx, y, tipo_moneda)
    y = draw_observaciones(pdf, observaciones, mx, y)
    y = draw_cierre(pdf, mx, y)
    y = draw_firma(pdf, "Luis Rojas Molina", "Jefe de Proyectos", "Snabbit Tecnologias", ancho, y)
    draw_footer(pdf, ancho, mx, my)
    draw_paginacion(pdf, ancho, mx, my)

    pdf.showPage()
    pdf.save()
    buffer.seek(0)
    return buffer.getvalue()

# ——— Wrapper para Jupyter ———
def generar_pdf_cotizacion_desde_model(cotizacion_id, ubicacion="Santiago"):
    from cotizaciones.models import Cotizacion
    cot = (
        Cotizacion.objects
        .select_related('empresa','cliente')
        .prefetch_related('items','solicitantes')
        .get(pk=cotizacion_id)
    )
    datos_cotizacion = {
        'numero_cotizacion': cot.numero_cotizacion,
        'descripcion': cot.descripcion or ''
    }
    datos_cotizacion['fecha_cotizacion'] = cot.fecha_creacion.strftime("%d de %B de %Y")

    # Logo
    logo_b64 = None
    logo_field = getattr(cot.empresa,'logo',None)
    logo_bytes = None
    if logo_field and hasattr(logo_field,'read'):
        try: logo_bytes = logo_field.read()
        except: logo_bytes=None
    elif isinstance(logo_field,str) and ',' in logo_field:
        logo_b64 = logo_field
    elif isinstance(logo_field,str):
        try:
            with open(logo_field,'rb') as f: logo_bytes=f.read()
        except: logo_bytes=None
    if logo_bytes:
        logo_b64='data:image/png;base64,'+base64.b64encode(logo_bytes).decode()

    # Solicitantes y items
    destinatarios = '/'.join(str(s.usuario) for s in cot.solicitantes.all()) or ''
    items = []
    for it in cot.items.all():
        items.append({
            'descripcion':     it.descripcion or it.nombre or '',
            'cantidad':        it.cantidad,
            'precio_unitario': f"{it.precio_unitario:.2f}",
            'total_neto':      f"{it.costo_total:.2f}"
        })

    # Contactos
    contactos = []
    if cot.empresa.telefono:   contactos.append(cot.empresa.telefono)
    if cot.empresa.email:      contactos.append(cot.empresa.email)
    if getattr(cot.empresa,'sitio_web',None): contactos.append(cot.empresa.sitio_web)

    return generar_pdf_cotizacion(
        datos_cotizacion, logo_b64,
        cot.empresa.nombre, getattr(cot.empresa,'rut_empresa',''), getattr(cot.empresa,'direccion_principal',''),
        cot.empresa.telefono or '', cot.empresa.email or '', getattr(cot.empresa,'sitio_web',''),
        cot.cliente.nombre, getattr(cot.cliente,'rut_empresa',''), getattr(cot.cliente,'direccion_principal',''),
        destinatarios, items, cot.observaciones or '', '', cot.empresa.nombre, '', contactos,
        ubicacion, cot.tipo_moneda
    )


In [86]:
%load_ext autoreload
%autoreload 2
from IPython.display import display, IFrame
pdf_bytes = generar_pdf_cotizacion_desde_model(3)
with open('cot_db.pdf', 'wb') as f:
    f.write(pdf_bytes)
display(IFrame('cot_db.pdf', width=1024, height=768))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [111]:
from io import BytesIO
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle, Paragraph
from reportlab.lib.utils import ImageReader
from reportlab.lib.styles import ParagraphStyle
from textwrap import wrap
import base64

# ——— Constantes de fuentes ———
FONTS = {
    'fecha':        ("Helvetica",        9),
    'titulo':       ("Helvetica-Bold",  14),
    'datos_label':  ("Helvetica-Bold",  10),
    'datos':        ("Helvetica",       10),
    'introduccion': ("Helvetica",       10),
    'tabla_head':   ("Helvetica-Bold",  10),
    'tabla':        ("Helvetica",       10),
    'fijo':         ("Helvetica-Oblique",9),
    'cierre':       ("Helvetica",       10),
    'firma_label':  ("Helvetica-Bold",  10),
    'firma':        ("Helvetica",        10),
    'firma_cargo':  ("Helvetica",        9),
    'footer':       ("Helvetica",        8),
}

# ——— Estilo para párrafo de introducción ———
INTRO_STYLE = ParagraphStyle(
    'intro', fontName='Helvetica', fontSize=10, leading=12
)

# ——— Estilo para pie de página ———
FOOTER_STYLE = ParagraphStyle(
    'footer', fontName='Helvetica', fontSize=8, leading=10, alignment=1
)

# ——— Funciones de dibujo ———
def draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my):
    pdf.setFont(*FONTS['fecha'])
    pdf.drawString(mx, alto - my + 1, f"{ubicacion}, {fecha_str}")

def draw_logo(pdf, logo_b64, ancho, alto, mx, my):
    if not logo_b64:
        return
    try:
        b64 = logo_b64.split(',', 1)[1] if ',' in logo_b64 else logo_b64
        img = ImageReader(BytesIO(base64.b64decode(b64)))
        iw, ih = img.getSize()
        w_logo = 120
        h_logo = w_logo * (ih / iw)
        pdf.drawImage(img, ancho - mx - w_logo, alto - my - h_logo,
                      width=w_logo, height=h_logo, mask='auto')
    except:
        pass

def draw_encabezado(pdf, ubicacion, fecha_str, logo_b64, ancho, alto, mx, my):
    draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my)
    draw_logo(pdf, logo_b64, ancho, alto, mx, my)

def draw_titulo(pdf, numero, ancho, alto, mx, my):
    pdf.setFont(*FONTS['titulo'])
    pdf.drawCentredString(ancho/2, alto - my - 40, f"Cotización N° {numero}")

def draw_datos_cliente(pdf, nombre, rut, direccion, destinatarios, mx, y):
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Cliente: {nombre}")
    y -= 14
    pdf.setFont(*FONTS['datos'])
    pdf.drawString(mx, y, f"Rut: {rut}")
    y -= 14
    pdf.drawString(mx, y, f"Dirección: {direccion}")
    y -= 20
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Estimado/a: {destinatarios}")
    return y - 20

def draw_introduccion(pdf, descripcion, mx, y, ancho):
    intro_text = (f"Ud. ha solicitado los precios de <b>{descripcion}</b>, "
                  "a continuación aparece nuestra cotización:")
    para = Paragraph(intro_text, INTRO_STYLE)
    w, h = para.wrap(ancho - 2 * mx, y)
    para.drawOn(pdf, mx, y - h)
    return y - h - 12

def draw_tabla_items(pdf, items, ancho, mx, y):
    data = [["Descripción", "Cantidad", "Precio Unit", "Total Neto"]]
    for it in items:
        data.append([it['descripcion'], str(it['cantidad']), it['precio_unitario'], it['total_neto']])
    table_width = ancho - 2 * mx
    colWidths = [0.4*table_width, 0.15*table_width, 0.2*table_width, 0.25*table_width]
    style = TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.lightgrey),
        ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
        ('ALIGN',      (1,1), (-1,-1), 'RIGHT'),
        ('GRID',       (0,0), (-1,-1), 0.5, colors.black),
        ('FONTSIZE',   (0,0), (-1,-1), 10),
    ])
    tabla = Table(data, colWidths=colWidths, style=style)
    w, h = tabla.wrapOn(pdf, 0, 0)
    tabla.drawOn(pdf, mx, y - h)
    return y - h - 20

def textotipomoneda(tipo_moneda):
    try:
        tipo = int(tipo_moneda)
    except:
        return ""
    if tipo == 1:
        return "Valores netos expresados en USD, conversión del dólar, observado del día de la compra +$5"
    elif tipo == 2:
        return "Valores netos expresados en CLP, debe agregar IVA"
    elif tipo == 3:
        return "Valores netos expresados en UF, debe agregar IVA"
    return ""

def draw_texto_fijo(pdf, mx, y, tipo_moneda):
    texto = textotipomoneda(tipo_moneda)
    pdf.setFont(*FONTS['fijo'])
    for line in wrap(texto, width=90):
        pdf.drawString(mx, y, line)
        y -= 12
    return y - 10

def draw_observaciones(pdf, observaciones, mx, y):
    if not observaciones:
        return y
    pdf.setFont(*FONTS['fijo'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(observaciones, width=90):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 20

def draw_cierre(pdf, mx, y):
    cierre = (
        "Gracias por darnos la oportunidad de ofrecerle este presupuesto. "
        "Como siempre, es para nosotros un placer hacer negocios con ustedes. "
        "Esperamos hacer realidad este pedido para su completa satisfacción."
    )
    pdf.setFont(*FONTS['cierre'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(cierre, width=115):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 30

# ——— Firma ———
#def draw_firma(pdf, firmante, cargo, ancho, y):
#    lines = ['Atentamente,', firmante]
#    if cargo:
#        lines.append(cargo)
#    for i, line in enumerate(lines):
#        font = FONTS['firma_label'] if i == 0 else FONTS['firma'] if i == 1 else FONTS['firma_cargo']
#        pdf.setFont(*font)
#        pdf.drawCentredString(ancho/2, y - i*14, line)
#    return y - len(lines)*14 - 10

def draw_firma(pdf, firmante, cargo, cargo2, ancho, y):
    lines = ['Atentamente,', firmante]
    if cargo:
        lines.append(cargo)
    if cargo2:
        lines.append(cargo2)
    for i, line in enumerate(lines):
        if i == 0:
            pdf.setFont(*FONTS['firma_label'])
        elif i == 1:
            pdf.setFont(*FONTS['firma'])
        else:
            pdf.setFont(*FONTS['firma_cargo'])
        pdf.drawCentredString(ancho/2, y - i*14, line)
    return y - len(lines)*14 - 10

# ——— Pie de página ———
def draw_footer(pdf, ancho, mx, my):
    # Texto con negrita al inicio usando Paragraph
    #footer_text = (
    #    "<b>Snabb IT</b> | Una empresa del Grupo A&amp;G<br/>"
    #    "Av. José Miguel Carrera N° 3840 Of- 808, San Miguel - Fono: (56-2) 27596140<br/>"
    #    "email: lrojas@snabb-it.cl administracion@aygasociados.cl<br/>"
    #    "Visítenos en <a href='https://snabbit.cl'>https://snabbit.cl</a>"
    #)
    footer_text = (
        "<b>Snabb IT | Asesores Tecnológicos</b><br/>"
        "<a href='https://maps.app.goo.gl/R1pAm1ANqs5eEjvSA'>Gran Av. José Miguel Carrera N° 3840 Of - 808, San Miguel</a> - Fono: <a href='tel:+56227596140'>(56-2) 27596140</a><br/>"
        "Visítenos en <a href='https://snabbit.cl' target='_blank'>https://snabbit.cl</a>"
    )
    para = Paragraph(footer_text, FOOTER_STYLE)
    available_width = ancho - 2 * mx
    w, h = para.wrap(available_width, my)
    para.drawOn(pdf, mx, my - h - 5)

# ——— Paginación ———
def draw_paginacion(pdf, ancho, mx, my):
    page = pdf.getPageNumber()
    total = page  # si no hay contador global, muestra x/x
    pdf.setFont(*FONTS['footer'])
    # Esquina inferior derecha
    y = 15
    pdf.drawRightString(ancho - mx, y, f"{page}/{total}")


def generar_pdf_cotizacion(
    datos_cotizacion, logo_base64, nombre_empresa, rut_empresa, direccion_empresa,
    telefono_empresa, email_empresa, sitio_web_empresa, nombre_cliente, rut_cliente,
    direccion_cliente, destinatarios, items=None, observaciones=None,
    cierre=None, firmante=None, cargo=None, cargo2=None, contactos=None,
    ubicacion="Santiago", tipo_moneda='1'):
    buffer = BytesIO(); pdf = canvas.Canvas(buffer, pagesize=A4)
    ancho, alto = A4; mx, my = 40, 40
    fecha_str = datos_cotizacion.get('fecha_cotizacion', datetime.now().strftime("%d de %B de %Y"))
    y = alto - my - 90
    draw_encabezado(pdf, ubicacion, fecha_str, logo_base64, ancho, alto, mx, my)
    draw_titulo(pdf, datos_cotizacion.get('numero_cotizacion',''), ancho, alto, mx, my)
    y = draw_datos_cliente(pdf, nombre_cliente, rut_cliente, direccion_cliente, destinatarios, mx, y)
    y = draw_introduccion(pdf, datos_cotizacion.get('descripcion',''), mx, y, ancho)
    y = draw_tabla_items(pdf, items or [], ancho, mx, y)
    y = draw_texto_fijo(pdf, mx, y, tipo_moneda)
    y = draw_observaciones(pdf, observaciones or '', mx, y)
    y = draw_cierre(pdf, mx, y)
    y = draw_firma(pdf, firmante or '', cargo or '', cargo2 or '', ancho, y)
    draw_footer(pdf, ancho, mx, my);
    draw_paginacion(pdf, ancho, mx, my);
    pdf.showPage(); pdf.save(); buffer.seek(0)
    return buffer.getvalue()


def generar_pdf_cotizacion_desde_model(cotizacion_id, ubicacion="Santiago"):
    from cotizaciones.models import Cotizacion
    cot = Cotizacion.objects.select_related('empresa','cliente').prefetch_related('items','solicitantes').get(pk=cotizacion_id)
    datos = {'numero_cotizacion': cot.numero_cotizacion, 'descripcion': cot.descripcion or ''}
    datos['fecha_cotizacion'] = cot.fecha_creacion.strftime("%d de %B de %Y")
    # Logo
    logo_b64 = None; logo_field = getattr(cot.empresa, 'logo', None); logo_bytes = None
    if logo_field and hasattr(logo_field,'read'):
        try: logo_bytes = logo_field.read()
        except: pass
    elif isinstance(logo_field, str) and ',' in logo_field:
        logo_b64 = logo_field
    elif isinstance(logo_field, str):
        try: logo_bytes = open(logo_field,'rb').read()
        except: pass
    if logo_bytes:
        logo_b64 = 'data:image/png;base64,' + base64.b64encode(logo_bytes).decode()
    destinatarios = '/'.join(str(s.usuario) for s in cot.solicitantes.all())
    items = [{'descripcion': it.descripcion or it.nombre or '', 'cantidad': it.cantidad,
              'precio_unitario': f"{it.precio_unitario:.2f}", 'total_neto': f"{it.costo_total:.2f}"}
             for it in cot.items.all()]
    contactos = [c for c in [cot.empresa.telefono, cot.empresa.email, cot.empresa.sitio_web] if c]
    return generar_pdf_cotizacion(
        datos_cotizacion=datos,
        logo_base64=logo_b64,
        nombre_empresa=cot.empresa.nombre,
        rut_empresa=getattr(cot.empresa,'rut_empresa',''),
        direccion_empresa=getattr(cot.empresa,'direccion_principal',''),
        telefono_empresa=cot.empresa.telefono,
        email_empresa=cot.empresa.email,
        sitio_web_empresa=getattr(cot.empresa,'sitio_web',''),
        nombre_cliente=cot.cliente.nombre,
        rut_cliente=getattr(cot.cliente,'rut_empresa',''),
        direccion_cliente=getattr(cot.cliente,'direccion_principal',''),
        destinatarios=destinatarios,
        items=items,
        observaciones=cot.observaciones or '',
        cierre='',
        firmante='Luis Rojas Molina',
        cargo='Jefe Proyectos',
        cargo2='Snabbit Tecnologías',
        contactos=contactos,
        ubicacion=ubicacion,
        tipo_moneda=cot.tipo_moneda
    )


In [113]:
%load_ext autoreload
%autoreload 2
from IPython.display import display, IFrame
pdf_bytes = generar_pdf_cotizacion_desde_model(3)
with open('cot_db.pdf', 'wb') as f:
    f.write(pdf_bytes)
display(IFrame('cot_db.pdf', width=1024, height=768))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [92]:
from cotizaciones.models import Cotizacion

In [93]:
for x in Cotizacion.objects.all():
    print(x.pk, x.tipo_moneda)

3 2
2 1
1 1


In [126]:
from io import BytesIO
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle, Paragraph
from reportlab.lib.utils import ImageReader
from reportlab.lib.styles import ParagraphStyle
from textwrap import wrap
import base64

# ——— Constantes de fuentes ———
FONTS = {
    'fecha':        ("Helvetica",        9),
    'titulo':       ("Helvetica-Bold",   14),
    'datos_label':  ("Helvetica-Bold",   10),
    'datos':        ("Helvetica",        10),
    'introduccion': ("Helvetica",        10),
    'tabla_head':   ("Helvetica-Bold",   10),
    'tabla':        ("Helvetica",        10),
    'fijo':         ("Helvetica-Oblique", 9),
    'cierre':       ("Helvetica",        10),
    'firma_label':  ("Helvetica-Bold",   10),
    'firma':        ("Helvetica",        10),
    'firma_cargo':  ("Helvetica",         9),
    'footer':       ("Helvetica",         8),
}

# ——— Estilo para párrafo de introducción ———
INTRO_STYLE = ParagraphStyle(
    'intro',
    fontName='Helvetica',
    fontSize=10,
    leading=12
)

# ——— Estilo para pie de página ———
FOOTER_STYLE = ParagraphStyle(
    'footer',
    fontName='Helvetica',
    fontSize=8,
    leading=10,
    alignment=1  # centrado
)

# ——— Funciones de dibujo ———
def draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my):
    pdf.setFont(*FONTS['fecha'])
    pdf.drawString(mx, alto - my + 1, f"{ubicacion}, {fecha_str}")

def draw_logo(pdf, logo_b64, ancho, alto, mx, my):
    if not logo_b64:
        return
    try:
        b64 = logo_b64.split(',', 1)[1] if ',' in logo_b64 else logo_b64
        img = ImageReader(BytesIO(base64.b64decode(b64)))
        iw, ih = img.getSize()
        w_logo = 120
        h_logo = w_logo * (ih / iw)
        pdf.drawImage(img, ancho - mx - w_logo, alto - my - h_logo,
                      width=w_logo, height=h_logo, mask='auto')
    except:
        pass

def draw_encabezado(pdf, ubicacion, fecha_str, logo_b64, ancho, alto, mx, my):
    draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my)
    draw_logo(pdf, logo_b64, ancho, alto, mx, my)

def draw_titulo(pdf, numero, ancho, alto, mx, my):
    pdf.setFont(*FONTS['titulo'])
    pdf.drawCentredString(ancho/2, alto - my - 40, f"Cotización N° {numero}")

def draw_datos_cliente(pdf, nombre, rut, direccion, destinatarios, mx, y):
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Cliente: {nombre}")
    y -= 14
    pdf.setFont(*FONTS['datos'])
    pdf.drawString(mx, y, f"Rut: {rut}")
    y -= 14
    pdf.drawString(mx, y, f"Dirección: {direccion}")
    y -= 20
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Estimado/a: {destinatarios}")
    return y - 20

def draw_introduccion(pdf, descripcion, mx, y, ancho):
    intro_text = (
        f"Ud. ha solicitado los precios de <b>{descripcion}</b>, "
        "a continuación aparece nuestra cotización:"
    )
    para = Paragraph(intro_text, INTRO_STYLE)
    w, h = para.wrap(ancho - 2 * mx, y)
    para.drawOn(pdf, mx, y - h)
    return y - h - 12

# ——— Tabla de ítems con formateo según moneda ———
def draw_tabla_items(pdf, items, ancho, mx, y, tipo_moneda):
    # Definir encabezados y formato según tipo_moneda
    try:
        tipo = int(tipo_moneda)
    except:
        tipo = 1

    if tipo == 1:
        encabezados = ["Descripción", "Cantidad", "Precio Unit USD", "Total Neto USD"]
    else:
        encabezados = ["Descripción", "Cantidad", "Precio Unit", "Total Neto"]

    data = [encabezados]
    for it in items:
        pu = float(it['precio_unitario'])
        tn = float(it['total_neto'])
        if tipo == 1:
            valor_pu = f"{pu:.2f} USD"
            valor_tn = f"{tn:.2f} USD"
        elif tipo == 2:
            valor_pu = f"${int(round(pu))}"
            valor_tn = f"${int(round(tn))}"
        else:
            valor_pu = f"{int(round(pu))} UF"
            valor_tn = f"{int(round(tn))} UF"

        data.append([
            it['descripcion'],
            str(it['cantidad']),
            valor_pu,
            valor_tn
        ])

    table_width = ancho - 2 * mx
    colWidths = [0.4 * table_width, 0.15 * table_width, 0.2 * table_width, 0.25 * table_width]
    style = TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('FONTNAME',   (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('ALIGN',      (1, 1), (-1, -1), 'RIGHT'),
        ('GRID',       (0, 0), (-1, -1), 0.5, colors.black),
        ('FONTSIZE',   (0, 0), (-1, -1), 10),
    ])
    tabla = Table(data, colWidths=colWidths, style=style)
    w, h = tabla.wrapOn(pdf, 0, 0)
    tabla.drawOn(pdf, mx, y - h)
    return y - h - 20

def textotipomoneda(tipo_moneda):
    try:
        tipo = int(tipo_moneda)
    except:
        return ""
    if tipo == 1:
        return "Valores netos expresados en USD, conversión del dólar, observado del día de la compra +$5"
    elif tipo == 2:
        return "Valores netos expresados en CLP, debe agregar IVA"
    elif tipo == 3:
        return "Valores netos expresados en UF, debe agregar IVA"
    return ""

def draw_texto_fijo(pdf, mx, y, tipo_moneda):
    texto = textotipomoneda(tipo_moneda)
    pdf.setFont(*FONTS['fijo'])
    for line in wrap(texto, width=90):
        pdf.drawString(mx, y, line)
        y -= 12
    return y - 10

def draw_observaciones(pdf, observaciones, mx, y):
    if not observaciones:
        return y
    pdf.setFont(*FONTS['fijo'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(observaciones, width=90):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 20

def draw_cierre(pdf, mx, y):
    cierre = (
        "Gracias por darnos la oportunidad de ofrecerle este presupuesto. "
        "Como siempre, es para nosotros un placer hacer negocios con ustedes. "
        "Esperamos hacer realidad este pedido para su completa satisfacción."
    )
    pdf.setFont(*FONTS['cierre'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(cierre, width=115):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 30

def draw_firma(pdf, firmante, cargo, cargo2, ancho, y):
    lines = ['Atentamente,', firmante]
    if cargo:
        lines.append(cargo)
    if cargo2:
        lines.append(cargo2)
    for i, line in enumerate(lines):
        if i == 0:
            pdf.setFont(*FONTS['firma_label'])
        elif i == 1:
            pdf.setFont(*FONTS['firma'])
        else:
            pdf.setFont(*FONTS['firma_cargo'])
        pdf.drawCentredString(ancho/2, y - i*14, line)
    return y - len(lines)*14 - 10

def draw_footer(pdf, ancho, mx, my):
    footer_text = (
        "<b>Snabb IT | Asesores Tecnológicos</b><br/>"
        "<a href='https://maps.app.goo.gl/R1pAm1ANqs5eEjvSA'>"
        "Gran Av. José Miguel Carrera N° 3840 Of - 808, San Miguel</a> - "
        "Fono: <a href='tel:+56227596140'>(56-2) 27596140</a><br/>"
        "Visítenos en <a href='https://snabbit.cl' target='_blank'>https://snabbit.cl</a>"
    )
    para = Paragraph(footer_text, FOOTER_STYLE)
    w, h = para.wrap(ancho - 2 * mx, my)
    para.drawOn(pdf, mx, my - h - 5)

def draw_paginacion(pdf, ancho, mx, my):
    page = pdf.getPageNumber()
    total = page
    pdf.setFont(*FONTS['footer'])
    pdf.drawRightString(ancho - mx, 15, f"{page}/{total}")

def generar_pdf_cotizacion(
    datos_cotizacion,
    logo_base64,
    nombre_empresa, rut_empresa, direccion_empresa,
    telefono_empresa, email_empresa, sitio_web_empresa,
    nombre_cliente, rut_cliente, direccion_cliente,
    destinatarios,
    items=None, observaciones=None,
    cierre=None, firmante=None, cargo=None, cargo2=None,
    contactos=None,
    ubicacion="Santiago", tipo_moneda='1'
):
    buffer = BytesIO()
    pdf = canvas.Canvas(buffer, pagesize=A4)
    ancho, alto = A4
    mx, my = 40, 40

    fecha_str = datos_cotizacion.get(
        'fecha_cotizacion',
        datetime.now().strftime("%d de %B de %Y")
    )

    y = alto - my - 90
    draw_encabezado(pdf, ubicacion, fecha_str, logo_base64, ancho, alto, mx, my)
    draw_titulo(pdf, datos_cotizacion.get('numero_cotizacion',''), ancho, alto, mx, my)
    y = draw_datos_cliente(pdf, nombre_cliente, rut_cliente, direccion_cliente, destinatarios, mx, y)
    y = draw_introduccion(pdf, datos_cotizacion.get('descripcion',''), mx, y, ancho)
    y = draw_tabla_items(pdf, items or [], ancho, mx, y, tipo_moneda)
    y = draw_texto_fijo(pdf, mx, y, tipo_moneda)
    y = draw_observaciones(pdf, observaciones or '', mx, y)
    y = draw_cierre(pdf, mx, y)
    y = draw_firma(pdf, firmante or '', cargo or '', cargo2 or '', ancho, y)
    draw_footer(pdf, ancho, mx, my)
    draw_paginacion(pdf, ancho, mx, my)

    pdf.showPage()
    pdf.save()
    buffer.seek(0)
    return buffer.getvalue()

def generar_pdf_cotizacion_desde_model(cotizacion_id, ubicacion="Santiago"):
    from cotizaciones.models import Cotizacion

    cot = (Cotizacion.objects
           .select_related('empresa','cliente')
           .prefetch_related('items','solicitantes')
           .get(pk=cotizacion_id))

    datos = {
        'numero_cotizacion': cot.numero_cotizacion,
        'descripcion': cot.descripcion or ''
    }
    datos['fecha_cotizacion'] = cot.fecha_creacion.strftime("%d de %B de %Y")

    # Logo
    logo_b64 = None
    logo_field = getattr(cot.empresa, 'logo', None)
    logo_bytes = None
    if logo_field and hasattr(logo_field, 'read'):
        try:
            logo_bytes = logo_field.read()
        except:
            pass
    elif isinstance(logo_field, str) and ',' in logo_field:
        logo_b64 = logo_field
    elif isinstance(logo_field, str):
        try:
            logo_bytes = open(logo_field, 'rb').read()
        except:
            pass
    if logo_bytes:
        logo_b64 = 'data:image/png;base64,' + base64.b64encode(logo_bytes).decode()

    destinatarios = '/'.join(str(s.usuario) for s in cot.solicitantes.all())
    items = [{
        'descripcion':     it.descripcion or it.nombre or '',
        'cantidad':        it.cantidad,
        'precio_unitario': f"{it.precio_unitario:.2f}",
        'total_neto':      f"{it.costo_total:.2f}"
    } for it in cot.items.all()]

    contactos = [c for c in [
        cot.empresa.telefono,
        cot.empresa.email,
        cot.empresa.sitio_web
    ] if c]

    return generar_pdf_cotizacion(
        datos_cotizacion=datos,
        logo_base64=logo_b64,
        nombre_empresa=cot.empresa.nombre,
        rut_empresa=getattr(cot.empresa, 'rut_empresa', ''),
        direccion_empresa=getattr(cot.empresa, 'direccion_principal', ''),
        telefono_empresa=cot.empresa.telefono,
        email_empresa=cot.empresa.email,
        sitio_web_empresa=getattr(cot.empresa, 'sitio_web', ''),
        nombre_cliente=cot.cliente.nombre,
        rut_cliente=getattr(cot.cliente, 'rut_empresa', ''),
        direccion_cliente=getattr(cot.cliente, 'direccion_principal', ''),
        destinatarios=destinatarios,
        items=items,
        observaciones=cot.observaciones or '',
        cierre='',
        firmante='Luis Rojas Molina',
        cargo='Jefe de Proyectos',
        cargo2='Snabbit Tecnologías',
        contactos=contactos,
        ubicacion=ubicacion,
        tipo_moneda=cot.tipo_moneda
    )


In [129]:
%load_ext autoreload
%autoreload 2
from IPython.display import display, IFrame
pdf_bytes = generar_pdf_cotizacion_desde_model(4)
with open('cot_db.pdf', 'wb') as f:
    f.write(pdf_bytes)
display(IFrame('cot_db.pdf', width=1024, height=768))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [130]:
from io import BytesIO
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle, Paragraph
from reportlab.lib.utils import ImageReader
from reportlab.lib.styles import ParagraphStyle
from textwrap import wrap
import base64

# ——— Constantes de fuentes ———
FONTS = {
    'fecha':        ("Helvetica",        9),
    'titulo':       ("Helvetica-Bold",  14),
    'datos_label':  ("Helvetica-Bold",  10),
    'datos':        ("Helvetica",       10),
    'introduccion': ("Helvetica",       10),
    'tabla_head':   ("Helvetica-Bold",  10),
    'tabla':        ("Helvetica",       10),
    'fijo':         ("Helvetica-Oblique",9),
    'cierre':       ("Helvetica",       10),
    'firma_label':  ("Helvetica-Bold",  10),
    'firma':        ("Helvetica",        10),
    'firma_cargo':  ("Helvetica",        9),
    'footer':       ("Helvetica",        8),
}

# ——— Estilo para párrafo de introducción ———
INTRO_STYLE = ParagraphStyle(
    'intro', fontName='Helvetica', fontSize=10, leading=12
)

# ——— Estilo para pie de página ———
FOOTER_STYLE = ParagraphStyle(
    'footer', fontName='Helvetica', fontSize=8, leading=10, alignment=1
)

# ——— Estilo para celdas de descripción ———
CELL_STYLE = ParagraphStyle(
    'cell', fontName='Helvetica', fontSize=10, leading=12
)

# ——— Funciones de dibujo ———
def draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my):
    pdf.setFont(*FONTS['fecha'])
    pdf.drawString(mx, alto - my + 1, f"{ubicacion}, {fecha_str}")

def draw_logo(pdf, logo_b64, ancho, alto, mx, my):
    if not logo_b64:
        return
    try:
        b64 = logo_b64.split(',', 1)[1] if ',' in logo_b64 else logo_b64
        img = ImageReader(BytesIO(base64.b64decode(b64)))
        iw, ih = img.getSize()
        w_logo = 120
        h_logo = w_logo * (ih / iw)
        pdf.drawImage(img, ancho - mx - w_logo, alto - my - h_logo,
                      width=w_logo, height=h_logo, mask='auto')
    except:
        pass

def draw_encabezado(pdf, ubicacion, fecha_str, logo_b64, ancho, alto, mx, my):
    draw_fecha(pdf, ubicacion, fecha_str, mx, alto, my)
    draw_logo(pdf, logo_b64, ancho, alto, mx, my)

def draw_titulo(pdf, numero, ancho, alto, mx, my):
    pdf.setFont(*FONTS['titulo'])
    pdf.drawCentredString(ancho/2, alto - my - 40, f"Cotización N° {numero}")

def draw_datos_cliente(pdf, nombre, rut, direccion, destinatarios, mx, y):
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Cliente: {nombre}")
    y -= 14
    pdf.setFont(*FONTS['datos'])
    pdf.drawString(mx, y, f"Rut: {rut}")
    y -= 14
    pdf.drawString(mx, y, f"Dirección: {direccion}")
    y -= 20
    pdf.setFont(*FONTS['datos_label'])
    pdf.drawString(mx, y, f"Estimado/a: {destinatarios}")
    return y - 20

def draw_introduccion(pdf, descripcion, mx, y, ancho):
    intro_text = (f"Ud. ha solicitado los precios de <b>{descripcion}</b>, "
                  "a continuación aparece nuestra cotización:")
    para = Paragraph(intro_text, INTRO_STYLE)
    w, h = para.wrap(ancho - 2 * mx, y)
    para.drawOn(pdf, mx, y - h)
    return y - h - 12

# ——— Tabla de ítems con formateo según moneda, wrapping y salto de página ———
def draw_tabla_items(pdf, items, ancho, mx, y, tipo_moneda):
    try:
        tipo = int(tipo_moneda)
    except:
        tipo = 1
    if tipo == 1:
        encabezados = ["Descripción", "Cantidad", "Precio Unit USD", "Total Neto USD"]
    else:
        encabezados = ["Descripción", "Cantidad", "Precio Unit", "Total Neto"]
    data = [encabezados]
    for it in items:
        desc = it['descripcion'].replace('■', '')
        desc_para = Paragraph(desc, CELL_STYLE)
        pu = float(it['precio_unitario'])
        tn = float(it['total_neto'])
        if tipo == 1:
            valor_pu = f"{pu:.2f} USD"
            valor_tn = f"{tn:.2f} USD"
        elif tipo == 2:
            valor_pu = f"${int(round(pu))}"
            valor_tn = f"${int(round(tn))}"
        else:
            valor_pu = f"{int(round(pu))} UF"
            valor_tn = f"{int(round(tn))} UF"
        data.append([desc_para, str(it['cantidad']), valor_pu, valor_tn])
    table_width = ancho - 2 * mx
    colWidths = [0.4 * table_width, 0.15 * table_width, 0.2 * table_width, 0.25 * table_width]
    style = TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('FONTNAME',   (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('ALIGN',      (1, 1), (-1, -1), 'RIGHT'),
        ('GRID',       (0, 0), (-1, -1), 0.5, colors.black),
        ('FONTSIZE',   (0, 0), (-1, -1), 10),
    ])
    tabla = Table(data, colWidths=colWidths, style=style, repeatRows=1)
    bottom_reserved = 200
    availW = table_width
    y_start = y
    remaining = tabla
    while True:
        availH = y_start - (40 + bottom_reserved)
        parts = remaining.split(availW, availH)
        part = parts[0]
        w, h = part.wrap(availW, availH)
        part.drawOn(pdf, mx, y_start - h)
        y_start -= h + 20
        if len(parts) > 1:
            pdf.showPage()
            draw_encabezado(pdf, ubicacion, fecha_str, logo_b64, ancho, alto, mx, my=40)
            draw_titulo(pdf, datos_cotizacion.get('numero_cotizacion',''), ancho, alto, mx, my=40)
            y_start = alto - 40 - 90
            remaining = parts[1]
            continue
        break
    return y_start

# ——— Resto de funciones ———
def textotipomoneda(tipo_moneda):
    try:
        tipo = int(tipo_moneda)
    except:
        return ""
    if tipo == 1:
        return "Valores netos expresados en USD, conversión del dólar, observado del día de la compra +$5"
    elif tipo == 2:
        return "Valores netos expresados en CLP, debe agregar IVA"
    elif tipo == 3:
        return "Valores netos expresados en UF, debe agregar IVA"
    return ""

def draw_texto_fijo(pdf, mx, y, tipo_moneda):
    texto = textotipomoneda(tipo_moneda)
    pdf.setFont(*FONTS['fijo'])
    for line in wrap(texto, width=90):
        pdf.drawString(mx, y, line)
        y -= 12
    return y - 10

def draw_observaciones(pdf, observaciones, mx, y):
    if not observaciones:
        return y
    pdf.setFont(*FONTS['fijo'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(observaciones, width=90):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 20

def draw_cierre(pdf, mx, y):
    cierre = (
        "Gracias por darnos la oportunidad de ofrecerle este presupuesto. "
        "Como siempre, es para nosotros un placer hacer negocios con ustedes. "
        "Esperamos hacer realidad este pedido para su completa satisfacción."
    )
    pdf.setFont(*FONTS['cierre'])
    text_obj = pdf.beginText(mx, y)
    for line in wrap(cierre, width=115):
        text_obj.textLine(line)
        y -= 12
    pdf.drawText(text_obj)
    return y - 30

def draw_firma(pdf, firmante, cargo, cargo2, ancho, y):
    lines = ['Atentamente,', firmante]
    if cargo:
        lines.append(cargo)
    if cargo2:
        lines.append(cargo2)
    for i, line in enumerate(lines):
        if i == 0:
            pdf.setFont(*FONTS['firma_label'])
        elif i == 1:
            pdf.setFont(*FONTS['firma'])
        else:
            pdf.setFont(*FONTS['firma_cargo'])
        pdf.drawCentredString(ancho/2, y - i*14, line)
    return y - len(lines)*14 - 10

def draw_footer(pdf, ancho, mx, my):
    footer_text = (
        "<b>Snabb IT | Asesores Tecnológicos</b><br/>"
        "<a href='https://maps.app.goo.gl/R1pAm1ANqs5eEjvSA'>Gran Av. José Miguel Carrera N° 3840 Of - 808, San Miguel</a> - "
        "Fono: <a href='tel:+56227596140'>(56-2) 27596140</a><br/>"
        "Visítenos en <a href='https://snabbit.cl' target='_blank'>https://snabbit.cl</a>"
    )
    para = Paragraph(footer_text, FOOTER_STYLE)
    w, h = para.wrap(ancho - 2 * mx, my)
    para.drawOn(pdf, mx, my - h - 5)

def draw_paginacion(pdf, ancho, mx, my):
    page = pdf.getPageNumber()
    total = page
    pdf.setFont(*FONTS['footer'])
    pdf.drawRightString(ancho - mx, 15, f"{page}/{total}")

# ——— Función principal ———
def generar_pdf_cotizacion(
    datos_cotizacion, logo_base64, nombre_empresa, rut_empresa, direccion_empresa,
    telefono_empresa, email_empresa, sitio_web_empresa, nombre_cliente, rut_cliente,
    direccion_cliente, destinatarios, items=None, observaciones=None,
    cierre=None, firmante=None, cargo=None, cargo2=None, contactos=None,
    ubicacion="Santiago", tipo_moneda='1'):
    buffer = BytesIO()
    pdf = canvas.Canvas(buffer, pagesize=A4)
    ancho, alto = A4
    mx, my = 40, 40
    fecha_str = datos_cotizacion.get(
        'fecha_cotizacion', datetime.now().strftime("%d de %B de %Y")
    )
    y = alto - my - 90
    draw_encabezado(pdf, ubicacion, fecha_str, logo_base64, ancho, alto, mx, my)
    draw_titulo(pdf, datos_cotizacion.get('numero_cotizacion',''), ancho, alto, mx, my)
    y = draw_datos_cliente(pdf, nombre_cliente, rut_cliente, direccion_cliente, destinatarios, mx, y)
    y = draw_introduccion(pdf, datos_cotizacion.get('descripcion',''), mx, y, ancho)
    y = draw_tabla_items(pdf, items or [], ancho, mx, y, tipo_moneda)
    y = draw_texto_fijo(pdf, mx, y, tipo_moneda)
    y = draw_observaciones(pdf, observaciones or '', mx, y)
    y = draw_cierre(pdf, mx, y)
    y = draw_firma(pdf, firmante or '', cargo or '', cargo2 or '', ancho, y)
    draw_footer(pdf, ancho, mx, my)
    draw_paginacion(pdf, ancho, mx, my)
    pdf.showPage()
    pdf.save()
    buffer.seek(0)
    return buffer.getvalue()

# ——— Wrapper para Django ———
def generar_pdf_cotizacion_desde_model(cotizacion_id, ubicacion="Santiago"):
    from cotizaciones.models import Cotizacion
    cot = (Cotizacion.objects
           .select_related('empresa','cliente')
           .prefetch_related('items','solicitantes')
           .get(pk=cotizacion_id))
    datos = {
        'numero_cotizacion': cot.numero_cotizacion,
        'descripcion': cot.descripcion or ''
    }
    datos['fecha_cotizacion'] = cot.fecha_creacion.strftime("%d de %B de %Y")
    # Logo
    logo_b64 = None
    logo_field = getattr(cot.empresa, 'logo', None)
    logo_bytes = None
    if logo_field and hasattr(logo_field, 'read'):
        try:
            logo_bytes = logo_field.read()
        except:
            pass
    elif isinstance(logo_field, str) and ',' in logo_field:
        logo_b64 = logo_field
    elif isinstance(logo_field, str):
        try:
            logo_bytes = open(logo_field, 'rb').read()
        except:
            pass
    if logo_bytes:
        logo_b64 = 'data:image/png;base64,' + base64.b64encode(logo_bytes).decode()
    destinatarios = '/'.join(str(s.usuario) for s in cot.solicitantes.all())
    items = [{
        'descripcion': it.descripcion or it.nombre or '',
        'cantidad': it.cantidad,
        'precio_unitario': f"{it.precio_unitario:.2f}",
        'total_neto': f"{it.costo_total:.2f}"
    } for it in cot.items.all()]
    contactos = [c for c in [cot.empresa.telefono, cot.empresa.email, cot.empresa.sitio_web] if c]
    return generar_pdf_cotizacion(
        datos_cotizacion=datos,
        logo_base64=logo_b64,
        nombre_empresa=cot.empresa.nombre,
        rut_empresa=getattr(cot.empresa, 'rut_empresa', ''),
        direccion_empresa=getattr(cot.empresa, 'direccion_principal', ''),
        telefono_empresa=cot.empresa.telefono,
        email_empresa=cot.empresa.email,
        sitio_web_empresa=getattr(cot.empresa, 'sitio_web', ''),
        nombre_cliente=cot.cliente.nombre,
        rut_cliente=getattr(cot.cliente, 'rut_empresa', ''),
        direccion_cliente=getattr(cot.cliente, 'direccion_principal', ''),
        destinatarios=destinatarios,
        items=items,
        observaciones=cot.observaciones or '',
        cierre='',
        firmante='Luis Rojas Molina',
        cargo='Jefe de Proyectos',
        cargo2='Snabbit Tecnologías',
        contactos=contactos,
        ubicacion=ubicacion,
        tipo_moneda=cot.tipo_moneda
    )


In [132]:
%load_ext autoreload
%autoreload 2
from IPython.display import display, IFrame
pdf_bytes = generar_pdf_cotizacion_desde_model(4)
with open('cot_db.pdf', 'wb') as f:
    f.write(pdf_bytes)
display(IFrame('cot_db.pdf', width=1024, height=768))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
